# Trabajo Práctico 3. Recomendador personalizado de libros

Procesamiento de Lenguaje Natural · TUIA · UNR

Grupo 2. Integrantes: Josías Calabozo, Sharo Giuntoli, Ismael Darruiz y Sebastián Di Carlo.

El notebook sigue los ocho pasos del enunciado, con sus mismos títulos. Lee los datos de la cátedra de `data/` y guarda los entregables en `entregables/`. Cómo armar el entorno y ejecutarlo está en el [README](README.md).

---
# Preparación

In [1]:
import os
import re
import unicodedata
import warnings
from collections import Counter
from itertools import combinations

# Sin avisos ni barras de progreso de Hugging Face en las salidas (va antes de importar sentence_transformers)
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
warnings.filterwarnings("ignore", message="IProgress not found")

import numpy as np
import pandas as pd
import spacy
from huggingface_hub.utils import logging as logging_hub
from langdetect import DetectorFactory, detect, detect_langs
from langdetect.lang_detect_exception import LangDetectException
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix, f1_score, precision_recall_curve,
                             precision_recall_fscore_support, precision_score, recall_score)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import (GroupKFold, RepeatedStratifiedKFold, cross_val_predict,
                                     cross_val_score, train_test_split)
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline
from thefuzz import fuzz
from transformers.utils import logging as logging_transformers

logging_hub.set_verbosity_error()
logging_transformers.disable_progress_bar()
pd.set_option("display.max_colwidth", None)   # las tablas muestran los textos completos

SEMILLA = 42
DetectorFactory.seed = SEMILLA   # langdetect también tiene una parte aleatoria

# Rutas relativas a P3/, la carpeta del notebook
RUTA_CATALOGO = "data/catalogo_tp3.jsonl"
RUTA_MUESTRA = "data/muestra_etiquetada.csv"
RUTA_LIMPIO = "data/catalogo_limpio.csv"
GRUPO = "grupo02"   # prefijo de los entregables que pide el enunciado: grupo 2
RUTA_SOSPECHOSOS = f"entregables/{GRUPO}_sospechosos.csv"
RUTA_PREFERENCIAS = f"entregables/{GRUPO}_preferencias.csv"

# Los embeddings que tardan en calcularse se guardan en embeddings/, que no se versiona
os.makedirs("embeddings", exist_ok=True)
RUTA_EMB_ORACIONES = "embeddings/oraciones_catalogo.npy"
RUTA_EMB_SINOPSIS = "embeddings/sinopsis_limpio.npy"

---
# Paso 1. Cargar y revisar los libros

Conservamos `slug` (identificador), `title` (para mostrar), `authors` y `synopsis` (texto de entrada).

In [2]:
catalogo = pd.read_json(RUTA_CATALOGO, lines=True)
libros = catalogo[["slug", "title", "authors", "synopsis"]].copy()
print(f"Catálogo de la cátedra: {len(libros)} libros\n")

# Las palabras en cursiva quedaron separadas por saltos de línea: cada secuencia de espacios en blanco
# pasa a ser un solo espacio. No se borra ningún otro carácter, porque algunos pueden delatar texto insertado
ejemplo = libros["synopsis"].str.contains("El primer americano\n", regex=False, na=False)
print("Antes:  ", repr(libros.loc[ejemplo, "synopsis"].iloc[0][:60]))
libros["synopsis"] = libros["synopsis"].fillna("").str.replace(r"\s+", " ", regex=True).str.strip()
print("Después:", repr(libros.loc[ejemplo, "synopsis"].iloc[0][:60]))

Catálogo de la cátedra: 8747 libros

Antes:   'El primer americano\n, ambiciosa obra que lleva camino de con'


Después: 'El primer americano , ambiciosa obra que lleva camino de con'


In [3]:
eliminados = {}   # motivo → cantidad de libros eliminados

sin_sinopsis = libros["synopsis"] == ""
eliminados["sin sinopsis"] = int(sin_sinopsis.sum())
libros = libros[~sin_sinopsis]

slug_repetido = libros["slug"].duplicated()   # marca la segunda aparición de cada slug, y las siguientes
eliminados["slug repetido"] = int(slug_repetido.sum())
libros = libros[~slug_repetido]
print(eliminados)

{'sin sinopsis': 1, 'slug repetido': 0}


## Duplicados exactos de sinopsis

Antes de borrarlos, miramos qué libros comparten exactamente la misma sinopsis.

In [4]:
# keep=False marca todas las apariciones de una sinopsis repetida, también la primera
repetidas = libros[libros["synopsis"].duplicated(keep=False)].copy()
repetidas["autores"] = repetidas["authors"].str.join(", ")
grupos = repetidas.groupby("synopsis").agg(libros=("slug", "size"), autores=("autores", "nunique"),
                                           ejemplo=("title", "first"))
print(f"{len(grupos)} sinopsis aparecen en más de un libro ({len(repetidas)} libros en total)")
grupos.sort_values("libros", ascending=False).reset_index(drop=True).head(8)

33 sinopsis aparecen en más de un libro (131 libros en total)


,libros,autores,ejemplo
0,58,1,Alemán – Unidad 40
1,7,7,Nueva Dimensión 7
2,4,1,Teatro crítico universal VII
3,3,1,Dionisíacas Cantos XXV-XXXVI
4,3,1,El fin de una cuadrilla
5,2,1,Prácticas sexuales alternativas (I)
6,2,1,Los años irreverentes. Escritos completos en El Amante/Cine Vol.2
7,2,1,Comunicaciones unificadas con Elastix


In [5]:
# Conservamos un libro por sinopsis: el primero que aparece en el catálogo
sinopsis_repetida = libros["synopsis"].duplicated()
eliminados["sinopsis repetida"] = int(sinopsis_repetida.sum())
libros = libros[~sinopsis_repetida].reset_index(drop=True)
print(f"Eliminados por sinopsis repetida: {eliminados['sinopsis repetida']}")

Eliminados por sinopsis repetida: 98


**Qué muestra**

Los grupos son cursos (las 58 unidades de «Alemán»), revistas (los 7 números de *Nueva Dimensión*, cada uno de un autor distinto) y obras en varios tomos, como *Teatro crítico universal*. Para un recomendador que lee sinopsis, los libros de cada grupo son indistinguibles: conservamos uno por grupo y eliminamos 98.

## Versiones casi iguales de una obra

Comparamos solo libros con los mismos autores. Los títulos se normalizan (minúsculas, sin puntuación y sin lo que está entre paréntesis, que suele ser la edición o la traducción), y los pares con `fuzz.token_set_ratio` de 85 o más son candidatos. Unimos un par si los títulos tienen los mismos números, arábigos o romanos (así «Conversaciones con Dios I» y «II» quedan separados), y además:

1. Los títulos normalizados son iguales: es otra edición o traducción, con su propia contratapa, así que la sinopsis no decide.
2. O un título contiene al otro (`token_set_ratio` de 100) y las sinopsis tienen una similitud de Jaccard de 0,7 o más; sin esa condición se unirían una saga y uno de sus tomos.

Ante la duda conservamos los dos: borrar un libro legítimo es peor que dejar dos versiones.

In [6]:
NUMEROS = re.compile(r"\b(?:\d+|[ivx]+)\b")   # números arábigos o romanos: "2", "1937", "iii"


def normalizar_titulo(titulo):
    """Minúsculas, sin lo que está entre paréntesis y sin puntuación."""
    titulo = titulo.lower()
    titulo = re.sub(r"\(.*?\)", " ", titulo)   # "Hamlet (Ed. bilingüe)" → "hamlet "
    titulo = re.sub(r"[^\w\s]", " ", titulo)   # "Ella. Hija de la sabiduría" → "ella  hija de la sabiduría"
    return re.sub(r"\s+", " ", titulo).strip()


def parecido_titulo(titulo_a, titulo_b):
    """token_set_ratio de los dos títulos normalizados, de 0 a 100."""
    return fuzz.token_set_ratio(normalizar_titulo(titulo_a), normalizar_titulo(titulo_b))


def mismos_numeros(titulo_a, titulo_b):
    """True si los dos títulos tienen los mismos números, también los que están entre paréntesis."""
    return NUMEROS.findall(titulo_a.lower()) == NUMEROS.findall(titulo_b.lower())


def titulo_igual(titulo_a, titulo_b):
    """True si los títulos normalizados son iguales y tienen los mismos números."""
    return mismos_numeros(titulo_a, titulo_b) and normalizar_titulo(titulo_a) == normalizar_titulo(titulo_b)


def titulo_contenido(titulo_a, titulo_b):
    """True si un título contiene todas las palabras del otro y los dos tienen los mismos números."""
    return mismos_numeros(titulo_a, titulo_b) and parecido_titulo(titulo_a, titulo_b) == 100


def palabras(texto):
    """Conjunto de palabras del texto, en minúsculas."""
    return set(re.findall(r"\w+", texto.lower()))


def jaccard(a, b):
    """Palabras en común sobre palabras en total."""
    return len(a & b) / len(a | b)

In [7]:
filas = []
# Un grupo por cada lista de autores: solo se comparan libros de los mismos autores
for autores, grupo in libros.groupby(libros["authors"].str.join(", ")):
    for i, j in combinations(grupo.index, 2):   # cada par de libros del grupo, una sola vez (i < j)
        titulo_i = libros.at[i, "title"]
        titulo_j = libros.at[j, "title"]
        parecido = parecido_titulo(titulo_i, titulo_j)
        if parecido >= 85:
            similitud = jaccard(palabras(libros.at[i, "synopsis"]), palabras(libros.at[j, "synopsis"]))
            filas.append({"i": i, "j": j, "titulo_1": titulo_i, "titulo_2": titulo_j, "parecido_titulo": parecido,
                          "titulo_igual": titulo_igual(titulo_i, titulo_j),
                          "titulo_contenido": titulo_contenido(titulo_i, titulo_j),
                          "similitud_sinopsis": round(similitud, 2)})
pares = pd.DataFrame(filas)
pares["unir"] = pares["titulo_igual"] | (pares["titulo_contenido"] & (pares["similitud_sinopsis"] >= 0.7))
print(f"Pares candidatos: {len(pares)}; con el mismo título: {pares['titulo_igual'].sum()}; se unen: {pares['unir'].sum()}")

Pares candidatos: 363; con el mismo título: 62; se unen: 68


In [8]:
# Pares con el mismo título: cuánto se parecen sus sinopsis
iguales = pares[pares["titulo_igual"]]
print(f"Similitud de sinopsis de los {len(iguales)} pares con el mismo título: mínima {iguales['similitud_sinopsis'].min()}, "
      f"mediana {iguales['similitud_sinopsis'].median()}")
iguales.sort_values("similitud_sinopsis")[["titulo_1", "titulo_2", "similitud_sinopsis"]].head(6)

Similitud de sinopsis de los 62 pares con el mismo título: mínima 0.07, mediana 0.165


,titulo_1,titulo_2,similitud_sinopsis
271,El juguete rabioso (edición crítica),El juguete rabioso,0.07
81,Fábulas (Ed. Clásicos a medida),Fábulas,0.08
88,La Celestina (Ilustrado),La Celestina (Castellano Moderno),0.10
9,Lazarillo de Tormes (ed. Francisco Rico),Lazarillo de Tormes (Ed. Clásicos a medida),0.10
97,Cuentos de Canterbury,Cuentos de Canterbury (Tr. Josefina Ferrer),0.10
35,El Imperio Final (Ed. ilustrada),El Imperio Final. (Ed. revisada),0.10


In [9]:
# Pares con un título contenido en el otro, pero no iguales, de mayor a menor similitud de sinopsis
contenidos = pares[pares["titulo_contenido"] & ~pares["titulo_igual"]]
contenidos.sort_values("similitud_sinopsis", ascending=False)[["titulo_1", "titulo_2", "similitud_sinopsis"]].head(12)

,titulo_1,titulo_2,similitud_sinopsis
261,Manual para viajeros por Andalucía y lectores en casa. Granada,Manual para viajeros por Andalucía y lectores en casa,1.00
171,Introducción a la ciencia. Una guía para todos (o casi),Introducción a la ciencia,1.00
141,El secreto del universo,El secreto del universo y otros ensayos científicos,0.99
207,El príncipe y el mendigo,Príncipe y mendigo,0.98
164,El contrato social,Contrato social,0.92
18,Sherlock Holmes,Sherlock Holmes. La colección completa,0.89
96,Fábulas,Fábulas morales,0.59
144,Harry Potter. La colección completa,Harry Potter (saga completa),0.36
173,Sansón agonista,Sonetos & Sansón agonista,0.33
234,Los detectives López y Baldosillo,Otra vez los detectives López y Baldosillo,0.30


**Qué muestra**

Las sinopsis de los 62 pares con el mismo título se parecen poco (Jaccard mediana de 0,165): son ediciones y traducciones, cada una con su contratapa. Si les pidiéramos sinopsis parecidas, quedarían casi todas separadas.

Entre los títulos contenidos hay un hueco: las mismas obras con o sin subtítulo, como «Sherlock Holmes» y «Sherlock Holmes. La colección completa», tienen 0,89 o más, y desde 0,59 para abajo aparecen libros distintos, como «Sansón agonista» y una compilación que lo incluye. El corte de 0,7 cae en ese hueco.

In [10]:
# Dos pares que unimos y dos que separamos
ejemplos = [("Romeo y Julieta", "Romeo y Julieta (trad. Neruda)"), ("El príncipe y el mendigo", "Príncipe y mendigo"),
            ("Harry Potter (saga completa)", "Harry Potter y el prisionero de Azkaban"),
            ("Tratados hipocráticos III", "Tratados hipocráticos II")]
es_ejemplo = [(titulo_1, titulo_2) in ejemplos for titulo_1, titulo_2 in zip(pares["titulo_1"], pares["titulo_2"])]
pares.loc[es_ejemplo, ["titulo_1", "titulo_2", "titulo_igual", "titulo_contenido", "similitud_sinopsis", "unir"]]

,titulo_1,titulo_2,titulo_igual,titulo_contenido,similitud_sinopsis,unir
135,Tratados hipocráticos III,Tratados hipocráticos II,False,False,0.82,False
146,Harry Potter (saga completa),Harry Potter y el prisionero de Azkaban,False,True,0.18,False
207,El príncipe y el mendigo,Príncipe y mendigo,False,True,0.98,True
361,Romeo y Julieta,Romeo y Julieta (trad. Neruda),True,True,0.12,True


In [11]:
# De cada par unido se conserva el libro que aparece primero en el catálogo (i) y se elimina el otro (j)
otra_version = set(pares.loc[pares["unir"], "j"])
eliminados["otra versión de la misma obra"] = len(otra_version)
libros = libros.drop(index=list(otra_version)).reset_index(drop=True)

print(f"Libros disponibles: {len(libros)}\n")
print("Eliminados por motivo:")
print(pd.Series(eliminados).to_string())
print(f"Total eliminados: {sum(eliminados.values())}")

Libros disponibles: 8588

Eliminados por motivo:
sin sinopsis                      1
slug repetido                     0
sinopsis repetida                98
otra versión de la misma obra    60
Total eliminados: 159


In [12]:
libros.head(5)

,slug,title,authors,synopsis
0,de-la-caverna-al-cosmos,De la caverna al cosmos,[Eudald Carbonell],"La Humanidad ha entrado en una nueva fase evolutiva. Esto implica cambios cada vez más rápidos, panoramas inciertos. Eudald Carbonell nos ayuda a visualizarlos y a entenderlos para que podamos afrontar el futuro con seguridad. La humanidad se transforma a pasos de gigante. Hechos que hasta ahora parecían ciencia ficción están a la orden del día: inteligencia artificial, nuevos órdenes sociales, viajes a otros planetas… Los humanos estamos constantemente construyendo escenarios y realidades que pueden parecer inconmensurables: ni siquiera la ciencia ficción puede llegar a adivinar lo que podemos llegar a imaginar y producir. Ante este mundo de posibilidades, Eudald Carbonell tiene claro que la historia, el conocimiento, la ciencia, la tecnología y el pensamiento crítico socializados pueden ser los referentes que nos aseguren el buen funcionamiento y el futuro como humanos. En esta obra, el eminente pensador pone orden a todos estos debates y busca darnos la clave con rigor y claridad."
1,quienes-somos-y-como-hemos-llegado-hasta-aqui,Quiénes somos y cómo hemos llegado hasta aquí,[David Reich],"La extracción de ADN de los huesos de nuestros ancestros ha permitido resolver importantes controversias sobre la prehistoria, hasta el punto de cambiar su interpretación. Hoy en día puede extraerse y analizarse ADN antiguo y, en buena medida gracias a las aportaciones del propio David Reich, se ha puesto de manifiesto que la genómica es tan relevante como la arqueología o la lingüística a la hora de estudiar las poblaciones del pasado. En Quiénes somos y cómo hemos llegado hasta aquí, Reich nos enseña que el genoma humano, además de aportar toda la información que necesita un óvulo humano fertilizado para desarrollarse, contiene en sí mismo la historia de nuestra especie. La revolución de la genómica y del ADN antiguo está transformando lo que sabemos sobre el linaje de los humanos y, según nos explica Reich, los estudios de ADN revelan una larga historia de desigualdad entre poblaciones distintas, así como entre ambos sexos y entre individuos de una misma población. Esta obra desmiente la interpretación ortodoxa de que no hay diferencias biológicas relevantes entre poblaciones humanas y, al mismo tiempo, utiliza los sólidos datos de la genómica para ilustrar la improbabilidad de que las diferencias, que sí existen, se correspondan con los estereotipos de siempre."
2,teotihuacan,Teotihuacan,[Eduardo Matos Moctezuma],"Teotihuacan es la primera gran ciudad del centro de México. Surgida en los albores de nuestra era, Teotihuacan dejará su influencia en diversas regiones de Mesoamérica. Fue contemporánea de ciudades tan importantes como Monte Albán, en Oaxaca; Palenque y Yaxchilán, en Chiapas, y Cholula, en Puebla. Durante siete siglos se desarrolló y fue la ciudad con mayor número de habitantes, en una extensión de 22 kilómetros cuadrados. La ciudad estaba dividida en dos ejes y en ella vemos los comienzos de toda una serie de características, tanto religiosas como de organización social, que posteriormente adoptarán las culturas que se asentaron en el centro de México. Ciudad cosmopolita, en ella se encontró presencia de otras culturas como la maya y la zapoteca. ¿Cómo surgió Teotihuacan? ¿Cuáles fueron los factores que determinaron su colapso hacia el año 750 d. C.? El arqueólogo Eduardo Matos Moctezuma nos lleva a lo largo de estas páginas por los senderos de esta urbe que marcó una serie de características que dejaron su huella en diversas sociedades mesoamericanas."
3,tiro-y-las-colonias-fenicias-de-occidente,Tiro y las colonias fenicias de Occidente,[M.ª Eugenia Aubet],"Entre los siglos VIII y VI a. C. los fenicios establecieron el primer sistema comercial mediterráneo, que abarcó desde el actual Líbano hasta la Península Ibérica y Marruecos. Mediante la fundación de una serie de colonias en las principales costas e islas del Mediterráneo

**Qué muestra**

Quedan 8.588 libros. Eliminamos 159: 1 sin sinopsis, ninguno por slug repetido, 98 por sinopsis repetida y 60 por ser otra versión de una obra.

Unimos «Romeo y Julieta» con la traducción de Neruda porque tienen el mismo título, y «El príncipe y el mendigo» con «Príncipe y mendigo» porque un título contiene al otro y las sinopsis son casi iguales. Separamos la saga de Harry Potter de uno de sus tomos, cuyas sinopsis se parecen poco, y los tomos II y III de *Tratados hipocráticos*: comparten la presentación de la colección, pero los números indican tomos distintos.

---
# Paso 2. Auditar el catálogo

Dividimos la muestra etiquetada por libro: el 70 % para desarrollar los detectores (elegir umbrales y revisar errores) y el 30 % para medirlos.

In [13]:
muestra = pd.read_csv(RUTA_MUESTRA, keep_default_na=False)
# La misma normalización de espacios que en el catálogo, para buscar sobre el mismo texto
for columna in ["synopsis", "fragmento"]:
    muestra[columna] = muestra[columna].str.replace(r"\s+", " ", regex=True).str.strip()

muestra_desarrollo, muestra_prueba = train_test_split(muestra, test_size=0.3, random_state=SEMILLA,
                                                      stratify=muestra["contaminado"])
limpias_desarrollo = muestra_desarrollo[muestra_desarrollo["contaminado"] == 0]
print(len(muestra_desarrollo), len(muestra_prueba))
print(f"Contaminados: {muestra_desarrollo['contaminado'].sum()} en desarrollo y {muestra_prueba['contaminado'].sum()} en prueba")

# Lo que marca cada detector en la parte de prueba: una columna por detector
marcas_prueba = pd.DataFrame(index=muestra_prueba.index)

350 150
Contaminados: 70 en desarrollo y 30 en prueba


In [14]:
# oraciones y medir son las funciones de la guía del enunciado. A oraciones le sumamos dos cortes: después
# de «»» y de «…», que en las sinopsis también cierran oraciones
def oraciones(texto):
    """Divide el texto en oraciones de 15 caracteres o más."""
    partes = re.split(r"(?<=[.!?»…])\s+|\n", texto)
    return [p.strip() for p in partes if len(p.strip()) >= 15]


def medir(nombre, marcado, datos):
    """Muestra la precisión y el recall de un detector."""
    y = datos["contaminado"]
    p = precision_score(y, marcado, zero_division=0)
    r = recall_score(y, marcado)
    print(f"{nombre}: precisión {p:.2f}, recall {r:.2f} ({marcado.sum()} libros marcados)")


def tabla_oraciones(datos):
    """Una fila por oración de cada sinopsis, con el slug de su libro."""
    filas = []
    for slug, texto in zip(datos["slug"], datos["synopsis"]):
        for oracion in oraciones(texto):
            filas.append((slug, oracion))
    return pd.DataFrame(filas, columns=["slug", "oracion"])

## 2.1 Expresiones regulares

Antes de escribir la regex miramos en qué terminan los dominios que aparecen en desarrollo, en los fragmentos insertados y en las sinopsis limpias.

In [15]:
# Dominios del tipo "nombre.terminación". (?!\.[a-z]) se queda con la última parte: de "www.tomoya.test", "test"
DOMINIO = r"\b[a-z0-9-]+\.([a-z]{2,})\b(?!\.[a-z])"
print("Fragmentos:", muestra_desarrollo["fragmento"].str.extractall(DOMINIO)[0].value_counts().to_dict())
print("Sinopsis limpias:", limpias_desarrollo["synopsis"].str.extractall(DOMINIO)[0].value_counts().to_dict())

Fragmentos: {'test': 12, 'invalid': 9, 'example': 7}
Sinopsis limpias: {'com': 1}


La única `.com` está en una sinopsis limpia, y las direcciones de los fragmentos usan terminaciones reservadas para ejemplos. Por eso un dominio suelto (sin `http` ni `www`) solo cuenta como enlace con `.test`, `.invalid` o `.example`. Los enlaces con `http` o `www`, los correos y los teléfonos se buscan con cualquier terminación.

In [16]:
URL = re.compile(
    r"https?://\S+"                                 # enlace con protocolo
    r"|www\.\S+"                                    # enlace que empieza con www.
    r"|\b[\w.-]+@[\w-]+\.[a-z]{2,}\b"               # correo: ventas@tomozona.test
    r"|\b[a-z0-9-]+\.(?:test|invalid|example)\b"    # dominio suelto con esas terminaciones: tomoya.test
    r"|\+\d[\d -]{8,}\d",                           # teléfono con código de país: +54 9 11 5217-8319
    re.IGNORECASE)


def detectar_regex(datos):
    """Marca las sinopsis donde la regex encuentra un enlace, un correo o un teléfono."""
    return datos["synopsis"].str.contains(URL)


marcas_prueba["regex"] = detectar_regex(muestra_prueba)
medir("regex", marcas_prueba["regex"], muestra_prueba)

regex: precisión 1.00, recall 0.40 (12 libros marcados)


In [17]:
# Falsos positivos en desarrollo: libros limpios con algo que la regex reconoce
marcado = detectar_regex(muestra_desarrollo)
falsos = muestra_desarrollo[marcado & (muestra_desarrollo["contaminado"] == 0)]
print("Falsos positivos en desarrollo:")
for titulo, texto in zip(falsos["title"], falsos["synopsis"]):
    print(f"  {titulo}: {URL.findall(texto)}")

print("\nAlgunos fragmentos que la regex no encuentra:")
no_encontrados = muestra_desarrollo[~marcado & (muestra_desarrollo["contaminado"] == 1)]
for fragmento in no_encontrados["fragmento"].drop_duplicates().head(8):
    print("  ", fragmento)

Falsos positivos en desarrollo:
  ¿Que pasaría si…?: ['http://www.xkcd.com']

Algunos fragmentos que la regex no encuentra:
   El libro más esperado del año llega por fin en edición digital completa. Leé este y miles de libros más sin publicidad. Activá tu prueba gratuita desde tu celular.
   Conseguí este título y más de 50.000 libros en la app LetraTotal: el primer mes es sin cargo.
   Si te gustó esta reseña, compartila con tus amigos y participá del sorteo de un lector electrónico.
   Entrá a librоexpress.invalid y leé sin límites.
   Ganá dinero desde tu casa recomendando libros. Cupos limitados, consultá requisitos.
   Leé este y miles de libros más sin publicidad. Activá tu prueba gratuita desde tu celular.
   ¡Oferta por tiempo limitado! Suscribite hoy y llevate tres libros de regalo.
   Link en nuestro sitio: h t t p s : / / epubhub [.] example


**Qué muestra**

La regex no se equivoca en prueba, pero encuentra menos de la mitad de los contaminados. Detecta bien los enlaces explícitos (con `http` o `www`), los correos y los teléfonos, y no detecta los enlaces disimulados (letras separadas, letras cirílicas, «[.]» en lugar de «.») ni el texto promocional sin dirección. Tampoco distingue un sitio real de uno promocional: el falso positivo de desarrollo menciona `www.xkcd.com`. Y un dominio `.com` sin `http` ni `www` se escaparía, porque los dominios sueltos se limitan a las terminaciones de los fragmentos.

## 2.2 Normalizar antes de buscar

Primero miramos qué caracteres raros hay en los fragmentos de desarrollo. `unicodedata` da el nombre y la categoría de cada carácter: la categoría `Cf` (*format*) agrupa los invisibles.

In [18]:
# Caracteres fuera del alfabeto latino (código mayor a 0x24F) o invisibles
raros = Counter()
for fragmento in muestra_desarrollo["fragmento"]:
    for c in fragmento:
        if ord(c) > 0x24F or unicodedata.category(c) == "Cf":
            raros[c] += 1
for c, n in raros.most_common():
    print(f"{c!r:10} {unicodedata.name(c, '?'):47} categoría {unicodedata.category(c)}  ({n} veces)")

# ¿Los invisibles aparecen también en sinopsis limpias?
con_invisibles = muestra_desarrollo["synopsis"].apply(lambda texto: any(unicodedata.category(c) == "Cf" for c in texto))
print("\nLibros con algún carácter invisible, según estén contaminados o no:")
print(pd.crosstab(con_invisibles, muestra_desarrollo["contaminado"], rownames=["con invisibles"]).to_string())

'\u200b'   ZERO WIDTH SPACE                                categoría Cf  (4 veces)
'\u2060'   WORD JOINER                                     categoría Cf  (3 veces)
'а'        CYRILLIC SMALL LETTER A                         categoría Ll  (2 veces)
'і'        CYRILLIC SMALL LETTER BYELORUSSIAN-UKRAINIAN I  categoría Ll  (2 veces)
'е'        CYRILLIC SMALL LETTER IE                        categoría Ll  (2 veces)
'о'        CYRILLIC SMALL LETTER O                         categoría Ll  (1 veces)
'с'        CYRILLIC SMALL LETTER ES                        categoría Ll  (1 veces)

Libros con algún carácter invisible, según estén contaminados o no:
contaminado       0   1
con invisibles         
False           276  63
True              4   7


Con eso escribimos `normalizar(texto)`, que deshace cada disfraz. Además de «punto» reemplazamos «dot», la misma palabra en inglés.

In [19]:
# Letras cirílicas que se ven iguales a las latinas: а е о р с у х і → a e o p c y x i
HOMOGLIFOS = str.maketrans("аеорсухі", "aeopcyxi")


def normalizar(texto):
    """Deshace los disfraces de los enlaces para que la regex los encuentre."""
    # 1. Caracteres invisibles: se borran todos los de la categoría Cf
    texto = "".join(c for c in texto if unicodedata.category(c) != "Cf")
    # 2. Letras de otro alfabeto que se ven iguales a las latinas
    texto = texto.translate(HOMOGLIFOS)
    # 3. Palabras o marcas en lugar de signos: "tomoya punto test", "tomoya dot test", "tomoya [.] test"
    texto = texto.replace("[.]", ".")
    texto = re.sub(r"\b(?:punto|dot)\b", ".", texto)
    texto = re.sub(r"\bbarra\b", "/", texto)
    texto = re.sub(r"\barroba\b", "@", texto)
    # 4. Letras sueltas separadas por espacios: "w w w" → "www", "h t t p s" → "https"
    texto = re.sub(r"(?<=\b\w) (?=\w\b)", "", texto)
    # 5. Sin espacios alrededor de los signos de una dirección: "libroplus . example" → "libroplus.example".
    #    También junta "fin. Luego", pero eso no crea enlaces falsos: la regex exige una terminación de dominio
    texto = re.sub(r"\s*([.:/@])\s*", r"\1", texto)
    return texto


def detectar_normalizacion(datos):
    """Marca las sinopsis donde la regex encuentra algo después de normalizar el texto."""
    return datos["synopsis"].apply(normalizar).str.contains(URL)


marcas_prueba["normalizacion"] = detectar_normalizacion(muestra_prueba)
medir("regex", marcas_prueba["regex"], muestra_prueba)
medir("regex + normalización", marcas_prueba["normalizacion"], muestra_prueba)

regex: precisión 1.00, recall 0.40 (12 libros marcados)


regex + normalización: precisión 1.00, recall 0.63 (19 libros marcados)


In [20]:
# Dos casos que la normalización recuperó: la regex no los encontraba en el texto original
recuperados = muestra_prueba[marcas_prueba["normalizacion"] & ~marcas_prueba["regex"]]
for fragmento in recuperados["fragmento"].head(2):
    print(f"Original:    {fragmento!r}")
    print(f"Normalizado: {normalizar(fragmento)!r}\n")

Original:    'Entrá a tоmoclub.example y leé sin límites.'
Normalizado: 'Entrá a tomoclub.example y leé sin límites.'

Original:    'Buscalo en w w w . novelalibre . example'
Normalizado: 'Buscalo en www.novelalibre.example'



**Qué muestra**

Los fragmentos disimulan las direcciones con caracteres invisibles de categoría Cf y con letras cirílicas iguales a las latinas. Los invisibles solos no son evidencia: 4 de los 280 libros limpios de desarrollo también tienen alguno. Por eso no marcamos un libro por tenerlos, sino que los borramos antes de buscar. La normalización sube el recall en prueba de 0,40 a 0,63 sin perder precisión; en el primer caso recuperado, la «о» de «tоmoclub» es cirílica aunque se vea igual.

## 2.3 Detectar el idioma de cada oración

Primero, el idioma de la sinopsis completa con `langdetect`.

In [21]:
idioma_sinopsis = muestra_desarrollo["synopsis"].apply(detect)
pd.crosstab(idioma_sinopsis, muestra_desarrollo["contaminado"], rownames=["idioma"])

contaminado,0,1
idioma,,
es,280,70


Ahora oración por oración, con `detect_langs`, que da la probabilidad de cada idioma. Para cada combinación de cantidad mínima de palabras y umbral de probabilidad, una sinopsis queda marcada si alguna de sus oraciones no está en español.

In [22]:
def idioma_mas_probable(oracion):
    """Idioma más probable de la oración y su probabilidad."""
    try:
        mejor = detect_langs(oracion)[0]   # detect_langs ordena los idiomas de mayor a menor probabilidad
        return mejor.lang, mejor.prob
    except LangDetectException:   # oraciones sin letras, como una lista de números
        return "?", 0.0


def con_idioma(orac):
    """La tabla de oraciones con dos columnas más: el idioma más probable y su probabilidad."""
    orac = orac.copy()
    resultados = [idioma_mas_probable(oracion) for oracion in orac["oracion"]]
    orac["idioma"] = [idioma for idioma, prob in resultados]
    orac["prob"] = [prob for idioma, prob in resultados]
    return orac


oraciones_desarrollo = con_idioma(tabla_oraciones(muestra_desarrollo))
oraciones_desarrollo["palabras"] = oraciones_desarrollo["oracion"].str.split().str.len()

filas = []
for minimo in [3, 5, 8, 10, 12]:
    for umbral in [0.5, 0.9, 0.99]:
        en_otro_idioma = oraciones_desarrollo[(oraciones_desarrollo["idioma"] != "es")
                                              & (oraciones_desarrollo["palabras"] >= minimo)
                                              & (oraciones_desarrollo["prob"] >= umbral)]
        marcado = muestra_desarrollo["slug"].isin(en_otro_idioma["slug"])
        filas.append({"mínimo de palabras": minimo, "umbral": umbral,
                      "precisión": precision_score(muestra_desarrollo["contaminado"], marcado, zero_division=0),
                      "recall": recall_score(muestra_desarrollo["contaminado"], marcado),
                      "marcados": marcado.sum()})
pd.DataFrame(filas).round(2)

,mínimo de palabras,umbral,precisión,recall,marcados
0,3,0.50,0.43,0.29,47
1,3,0.90,0.41,0.17,29
2,3,0.99,0.41,0.17,29
3,5,0.50,0.59,0.27,32
4,5,0.90,0.57,0.17,21
5,5,0.99,0.57,0.17,21
6,8,0.50,0.68,0.19,19
7,8,0.90,0.73,0.11,11
8,8,0.99,0.73,0.11,11
9,10,0.50,0.76,0.19,17


In [23]:
# Oraciones de menos de 10 palabras que no salen en español
cortas = oraciones_desarrollo[(oraciones_desarrollo["idioma"] != "es") & (oraciones_desarrollo["palabras"] < 10)]
cortas[["oracion", "palabras", "idioma", "prob"]].head(10).round(2)

,oracion,palabras,idioma,prob
51,Activá tu prueba gratuita desde tu celular.,7,ro,1.00
69,Nelly Kaprièlian Les Inrockuptibles,4,en,0.57
77,¿Por qué existe este caos?,5,fr,0.57
98,¿SERÁS EL MAGO O EL GUERRERO?,6,pt,1.00
102,¿LA MAGIA O LA ESPADA?,5,pt,1.00
198,El amor nunca deja de ser.,6,ca,0.57
330,Los Angeles Book Time Review,5,af,1.00
432,Activá tu prueba gratuita desde tu celular.,7,ro,1.00
724,Cuando las «Wallflowers»,3,en,1.00
739,pero nunca llega.,3,it,0.63


**Qué muestra**

Con la sinopsis completa, langdetect dice «español» para los 350 libros de desarrollo, también para los 70 contaminados: el mensaje insertado es una o dos oraciones dentro de un texto largo en español.

Por oración, las cortas dan falsos positivos (con 3 palabras como mínimo, la precisión es 0,43), como «¿LA MAGIA O LA ESPADA?», que sale como portugués. Elegimos 10 palabras y umbral 0,9, la combinación con mayor precisión en desarrollo (0,80). El costo es perder mensajes cortos reales, como «Activá tu prueba gratuita desde tu celular.», que sale como rumano.

In [24]:
MIN_PALABRAS = 10
UMBRAL_IDIOMA = 0.9


def oraciones_en_otro_idioma(datos):
    """Las oraciones de MIN_PALABRAS o más que no están en español con probabilidad UMBRAL_IDIOMA o más."""
    orac = tabla_oraciones(datos)
    orac = con_idioma(orac[orac["oracion"].str.split().str.len() >= MIN_PALABRAS])   # las cortas ni se analizan
    return orac[(orac["idioma"] != "es") & (orac["prob"] >= UMBRAL_IDIOMA)]


def detectar_idioma(datos):
    """Marca las sinopsis con alguna oración en otro idioma."""
    return datos["slug"].isin(oraciones_en_otro_idioma(datos)["slug"])


marcas_prueba["idioma"] = detectar_idioma(muestra_prueba)
medir("idioma por oración", marcas_prueba["idioma"], muestra_prueba)

idioma por oración: precisión 0.89, recall 0.27 (9 libros marcados)


**Qué muestra**

En prueba, 8 de los 9 libros que marca están contaminados. El recall es bajo porque solo encuentra la contaminación que está en otro idioma.

## 2.4 Reconocer entidades

Aplicamos NER con spaCy (`es_core_news_lg`) a las oraciones de los fragmentos de desarrollo y a la misma cantidad de oraciones de sinopsis limpias, elegidas al azar.

In [25]:
nlp = spacy.load("es_core_news_lg")

oraciones_fragmento = []
for fragmento in muestra_desarrollo.loc[muestra_desarrollo["contaminado"] == 1, "fragmento"]:
    oraciones_fragmento += oraciones(fragmento)
oraciones_limpias = tabla_oraciones(limpias_desarrollo)["oracion"].sample(n=len(oraciones_fragmento),
                                                                           random_state=SEMILLA)


def contar_entidades(textos):
    """Cuenta las entidades por etiqueta y por texto."""
    por_etiqueta, por_texto = Counter(), Counter()
    for doc in nlp.pipe(textos):   # nlp.pipe procesa muchos textos de una vez
        for ent in doc.ents:
            por_etiqueta[ent.label_] += 1
            por_texto[f"{ent.text} ({ent.label_})"] += 1
    return por_etiqueta, por_texto


etiquetas_fragmento, textos_fragmento = contar_entidades(oraciones_fragmento)
etiquetas_limpias, textos_limpios = contar_entidades(oraciones_limpias)
print(f"{len(oraciones_fragmento)} oraciones de cada grupo\n")
print(pd.DataFrame({"fragmentos": etiquetas_fragmento, "limpias": etiquetas_limpias}).fillna(0).astype(int).to_string())
print("\nEntidades más frecuentes en los fragmentos:", textos_fragmento.most_common(12))
print("\nEntidades más frecuentes en las oraciones limpias:", textos_limpios.most_common(12))

88 oraciones de cada grupo

      fragmentos  limpias
MISC          37       21
LOC           15       17
PER           16       36
ORG            2        4

Entidades más frecuentes en los fragmentos: [('compartila (PER)', 7), ('PDF (MISC)', 6), ('Ganá (PER)', 5), ('Download this ebook for FREE (MISC)', 4), ('sign-up required (MISC)', 4), ('EPUB sin registro (MISC)', 4), ('Descargalo (MISC)', 4), ('Oferta (LOC)', 2), ('EPUB (MISC)', 2), ('LibroLibre (MISC)', 2), ('EpubMax (MISC)', 2), ('Visitá pagin\u2060azona.invalid (LOC)', 1)]

Entidades más frecuentes en las oraciones limpias: [('IMPORTANTE (MISC)', 2), ('PDF (MISC)', 2), ('México (LOC)', 2), ('Mariana (PER)', 2), ('Fred (PER)', 1), ('Pug Henry (PER)', 1), ('Pamela (PER)', 1), ('Alejandro de Macedonia (PER)', 1), ('Star Wars (MISC)', 1), ('Graham Greene (PER)', 1), ('cinturón de Venus (LOC)', 1), ('Harold Lawlor (PER)', 1)]


**Qué muestra**

En las oraciones limpias predominan las personas (36: autores y personajes). En los fragmentos predomina MISC (37 de 70 entidades): formatos (*PDF*, *EPUB*), marcas inventadas (*LibroLibre*, *EpubMax*) y frases en inglés (*Download this ebook for FREE*). Las «personas» de los fragmentos son errores: «Ganá» y «compartila» son verbos con mayúscula al comienzo de la oración.

¿Qué entidades aparecen en el texto insertado y no en una sinopsis normal? Formatos de archivo, marcas o aplicaciones y texto en otro idioma etiquetado como MISC.

¿Sirven para detectar o solo para explicar? Para explicar. MISC también es frecuente en las oraciones limpias (21), incluso *PDF*, así que una regla como «tiene MISC» marcaría muchos libros legítimos. En un libro que ya marcó otro detector, las entidades muestran qué marca o qué formato se promociona.

## 2.5 Clasificar oraciones

Armamos un conjunto de oraciones con los libros de desarrollo: las del fragmento son positivas y las de las sinopsis limpias, negativas (el resto del texto de los libros contaminados no se usa). Entrenamos dos clasificadores con `LogisticRegression(class_weight="balanced", max_iter=1000)`: uno con TF-IDF y otro con los embeddings de `multilingual-e5-small`, uno de los modelos de oración multilingües del TP2.

In [26]:
filas = []
for slug, sinopsis, contaminado, fragmento in zip(muestra_desarrollo["slug"], muestra_desarrollo["synopsis"],
                                                  muestra_desarrollo["contaminado"], muestra_desarrollo["fragmento"]):
    if contaminado == 1:
        filas += [(slug, oracion, 1) for oracion in oraciones(fragmento)]
    else:
        filas += [(slug, oracion, 0) for oracion in oraciones(sinopsis)]
oraciones_entrenamiento = pd.DataFrame(filas, columns=["slug", "oracion", "spam"])
y_oraciones = oraciones_entrenamiento["spam"]
distintas = oraciones_entrenamiento.loc[y_oraciones == 1, "oracion"].nunique()
print(f"{len(oraciones_entrenamiento)} oraciones: {y_oraciones.sum()} positivas ({distintas} distintas)")

1940 oraciones: 88 positivas (62 distintas)


In [27]:
e5 = SentenceTransformer("intfloat/multilingual-e5-small")


def codificar(textos):
    """Embeddings normalizados de E5 para una lista de textos."""
    # E5 pide un prefijo según el uso: "query: " para clasificar o comparar textos entre sí.
    # En el TP2 usamos "passage: " porque ahí los libros se buscaban con consultas
    return e5.encode(["query: " + texto for texto in textos], normalize_embeddings=True, batch_size=64)


def codificar_con_cache(textos, ruta):
    """Como codificar, pero guarda el resultado en ruta y, si ya existe, lo lee de ahí."""
    if os.path.exists(ruta):
        E = np.load(ruta)
    else:
        E = codificar(textos)
        np.save(ruta, E)
    assert len(E) == len(textos), f"{ruta} no corresponde a estos textos: borrarlo y volver a ejecutar"
    return E


E_oraciones_entrenamiento = codificar(oraciones_entrenamiento["oracion"].tolist())

El puntaje de un libro es la probabilidad de su oración más sospechosa. Para cada clasificador elegimos con `precision_recall_curve`, sobre los libros de desarrollo, el umbral más bajo que alcanza la precisión buscada. Esas probabilidades no pueden salir de un modelo que ya vio la oración: `cross_val_predict` predice cada una con un modelo entrenado con las otras particiones. Los mensajes insertados son plantillas que se repiten en varios libros (de las 88 oraciones positivas, solo 62 son distintas), así que `GroupKFold` arma las particiones por el texto de la oración: cada oración repetida queda entera en una sola partición. Las variantes de una plantilla que solo cambian el nombre del sitio sí pueden quedar en particiones distintas: para esas, la validación es algo optimista.

Buscamos una precisión de 0,97 y no de 0,90: la muestra tiene un 20 % de contaminados y el catálogo bastante menos, así que ahí la misma regla va a marcar más libros legítimos por cada contaminado. No pedimos más porque, con 70 contaminados en desarrollo, un solo falso positivo deja la precisión debajo de 0,99.

In [28]:
PRECISION_BUSCADA = 0.97


def puntaje_por_libro(orac, columna, datos):
    """Puntaje de cada libro de datos: la probabilidad (columna) de su oración más sospechosa, o 0 si no tiene oraciones."""
    maximos = orac.groupby("slug")[columna].max()            # la oración más sospechosa de cada libro
    maximos = maximos.reindex(datos["slug"], fill_value=0)   # en el orden de datos, con 0 si el libro no tiene oraciones
    return pd.Series(maximos.to_numpy(), index=datos.index)


def umbral_por_precision(y, puntajes, precision_buscada):
    """El umbral más bajo que alcanza la precisión buscada, con su precisión y su recall."""
    precision, recall, umbrales = precision_recall_curve(y, puntajes)
    # precision y recall tienen un valor más que umbrales (el último no tiene umbral): se descarta
    alcanza = list(precision[:-1] >= precision_buscada)
    i = alcanza.index(True)   # el primero que alcanza, porque los umbrales van de menor a mayor
    return umbrales[i], precision[i], recall[i]

In [29]:
# Con TF-IDF se usa un pipeline, para que el vectorizador se ajuste dentro de cada partición
clasificador_tfidf = make_pipeline(TfidfVectorizer(), LogisticRegression(class_weight="balanced", max_iter=1000))
clasificador_embeddings = LogisticRegression(class_weight="balanced", max_iter=1000)
particiones = GroupKFold(n_splits=5)
grupo_de_cada_oracion = oraciones_entrenamiento["oracion"]   # cada oración repetida queda en una sola partición

# Probabilidad de spam de cada oración según un modelo que no la vio (la columna 1 de predict_proba es la clase spam)
oraciones_entrenamiento["prob_tfidf"] = cross_val_predict(
    clasificador_tfidf, oraciones_entrenamiento["oracion"], y_oraciones, groups=grupo_de_cada_oracion,
    cv=particiones, method="predict_proba")[:, 1]
oraciones_entrenamiento["prob_embeddings"] = cross_val_predict(
    clasificador_embeddings, E_oraciones_entrenamiento, y_oraciones, groups=grupo_de_cada_oracion,
    cv=particiones, method="predict_proba")[:, 1]

# Los modelos finales se entrenan con todas las oraciones de desarrollo
clasificador_tfidf.fit(oraciones_entrenamiento["oracion"], y_oraciones)
clasificador_embeddings.fit(E_oraciones_entrenamiento, y_oraciones)

y_desarrollo = muestra_desarrollo["contaminado"]
puntajes_tfidf = puntaje_por_libro(oraciones_entrenamiento, "prob_tfidf", muestra_desarrollo)
umbral_tfidf, p, r = umbral_por_precision(y_desarrollo, puntajes_tfidf, PRECISION_BUSCADA)
print(f"TF-IDF: umbral {umbral_tfidf:.3f} (en desarrollo: precisión {p:.2f}, recall {r:.2f}). "
      f"Con 0,5 marcaría {(puntajes_tfidf >= 0.5).sum()} libros")

puntajes_embeddings = puntaje_por_libro(oraciones_entrenamiento, "prob_embeddings", muestra_desarrollo)
umbral_embeddings, p, r = umbral_por_precision(y_desarrollo, puntajes_embeddings, PRECISION_BUSCADA)
print(f"Embeddings: umbral {umbral_embeddings:.3f} (en desarrollo: precisión {p:.2f}, recall {r:.2f}). "
      f"Con 0,5 marcaría {(puntajes_embeddings >= 0.5).sum()} libros")

TF-IDF: umbral 0.706 (en desarrollo: precisión 0.98, recall 0.77). Con 0,5 marcaría 59 libros
Embeddings: umbral 0.774 (en desarrollo: precisión 0.98, recall 0.79). Con 0,5 marcaría 83 libros


In [30]:
# Probabilidad de spam de cada oración de los libros de prueba, con los modelos finales
oraciones_prueba = tabla_oraciones(muestra_prueba)
oraciones_prueba["prob_tfidf"] = clasificador_tfidf.predict_proba(oraciones_prueba["oracion"])[:, 1]
E_oraciones_prueba = codificar(oraciones_prueba["oracion"].tolist())
oraciones_prueba["prob_embeddings"] = clasificador_embeddings.predict_proba(E_oraciones_prueba)[:, 1]

# Un libro es sospechoso si su oración más sospechosa alcanza el umbral del clasificador
puntajes_prueba_tfidf = puntaje_por_libro(oraciones_prueba, "prob_tfidf", muestra_prueba)
puntajes_prueba_embeddings = puntaje_por_libro(oraciones_prueba, "prob_embeddings", muestra_prueba)
marcas_prueba["clasificador_tfidf"] = puntajes_prueba_tfidf >= umbral_tfidf
marcas_prueba["clasificador_embeddings"] = puntajes_prueba_embeddings >= umbral_embeddings
medir("clasificador TF-IDF", marcas_prueba["clasificador_tfidf"], muestra_prueba)
medir("clasificador de embeddings", marcas_prueba["clasificador_embeddings"], muestra_prueba)

clasificador TF-IDF: precisión 1.00, recall 0.90 (27 libros marcados)
clasificador de embeddings: precisión 0.95, recall 0.60 (19 libros marcados)


In [31]:
# Los contaminados de prueba, según qué clasificador los encuentra
contaminados = muestra_prueba["contaminado"] == 1
marca_tfidf = marcas_prueba["clasificador_tfidf"]
marca_embeddings = marcas_prueba["clasificador_embeddings"]
print(pd.crosstab(marca_tfidf[contaminados], marca_embeddings[contaminados], rownames=["TF-IDF"], colnames=["embeddings"]))

print("\nLos que encuentra TF-IDF y no el de embeddings:")
for fragmento in muestra_prueba.loc[contaminados & marca_tfidf & ~marca_embeddings, "fragmento"]:
    print("  ", fragmento[:110])

print("\nLos que encuentra el de embeddings y no TF-IDF:")
for fragmento in muestra_prueba.loc[contaminados & ~marca_tfidf & marca_embeddings, "fragmento"]:
    print("  ", fragmento[:110])

embeddings  False  True 
TF-IDF                  
False           2      1
True           10     17

Los que encuentra TF-IDF y no el de embeddings:
   Una obra imprescindible que no puede faltar en tu biblioteca personal. Más títulos como este en www.librolibre
   Entrá a tоmoclub.example y leé sin límites.
   ¡Oferta por tiempo limitado! Suscribite hoy y llevate tres libros de regalo.
   Buscalo en w w w . novelalibre . example
   Si te gustó esta reseña, compartila con tus amigos y participá del sorteo de un lector electrónico.
   Buscalo en w w w . letramax . example
   Descargalo gratis en TomoMax punto example barra cardamomo
   Pedí tu copia escribiendo a ventas@letrazona.invalid
   ¡Oferta por tiempo limitado! Suscribite hoy y llevate tres libros de regalo.
   Una historia apasionante que te atrapará desde la primera página hasta el final. Descargá el libro completo gr

Los que encuentra el de embeddings y no TF-IDF:
   Escribinos por WhatsApp al +54 9 11 5452-7595 y recibí el 

In [32]:
# Falsos positivos de prueba: libros limpios que marca algún clasificador, con su oración más sospechosa
for nombre in ["tfidf", "embeddings"]:
    falsos = muestra_prueba.loc[marcas_prueba[f"clasificador_{nombre}"] & ~contaminados, "slug"]
    print(f"Falsos positivos del clasificador {nombre}: {len(falsos)}")
    orac = oraciones_prueba[oraciones_prueba["slug"].isin(falsos)]
    # Se ordena por probabilidad y se deja la primera oración de cada libro
    for _, fila in orac.sort_values(f"prob_{nombre}", ascending=False).drop_duplicates("slug").iterrows():
        print(f"  {fila['slug']} ({fila[f'prob_{nombre}']:.2f}): {fila['oracion'][:150]}")

Falsos positivos del clasificador tfidf: 0
Falsos positivos del clasificador embeddings: 1
  diario-emprendedor (0.80): que encontrarás al final de cada capítulo.


**Qué muestra**

A igual precisión en desarrollo, TF-IDF encuentra más contaminados de prueba y no tiene falsos positivos; el de embeddings marca *Diario emprendedor* por una oración que le habla al lector en segunda persona, como los fragmentos. Con un umbral de 0,5, TF-IDF marcaría 59 libros de desarrollo y el de embeddings, 83, para 70 contaminados: el mismo umbral no significa lo mismo en los dos modelos, y por eso cada uno tiene el suyo.

¿Por qué clasificar oraciones y no la sinopsis completa? El mensaje insertado es una o dos oraciones dentro de un texto legítimo, y en la sinopsis completa su señal se diluye, como le pasó a langdetect.

¿Qué representación detecta mejor los contaminados de prueba? TF-IDF: encuentra 10 que el de embeddings no encuentra, por cadenas literales («ventas@», «punto example barra»), direcciones disimuladas y frases que repiten las de desarrollo, como «¡Oferta por tiempo limitado!». El de embeddings encuentra uno solo que TF-IDF no, un pedido por WhatsApp, que de todos modos tiene un teléfono que encuentra la regex.

¿Cuál debería generalizar mejor a frases nuevas con el mismo sentido? La de embeddings, porque representa el sentido de la oración: una frase promocional nueva queda cerca de las de la muestra aunque no comparta palabras con ellas. TF-IDF solo reconoce palabras que ya vio, y por eso detecta mejor las cadenas literales, como una URL.

## 2.6 Encontrar libros copiados

Para cada libro del catálogo buscamos su vecino más parecido con `NearestNeighbors` (similitud del coseno sobre TF-IDF) y revisamos los pares con similitud alta y ningún autor en común. En cada par, la copia es la sinopsis más larga, porque tiene el texto del original más lo que le agregaron. Confirmamos cada par con Jaccard, como en el paso 1: una copia tiene casi todas las palabras del original, y dos libros de una serie comparten la presentación pero no la trama.

In [33]:
X_catalogo = TfidfVectorizer().fit_transform(libros["synopsis"])
vecinos = NearestNeighbors(n_neighbors=2, metric="cosine").fit(X_catalogo)
distancias, indices = vecinos.kneighbors(X_catalogo)
# La columna 0 es el propio libro (distancia 0) y la 1, su vecino más parecido
libros["vecino"] = indices[:, 1]
libros["similitud_vecino"] = 1 - distancias[:, 1]   # distancia coseno = 1 - similitud

In [34]:
# Cada libro con su vecino, si no tienen ningún autor en común (palabras y jaccard son las funciones del paso 1)
filas = []
for i, j in enumerate(libros["vecino"]):
    autores_en_comun = set(libros.at[i, "authors"]) & set(libros.at[j, "authors"])
    if len(autores_en_comun) == 0:
        # La copia es la de sinopsis más larga
        if len(libros.at[i, "synopsis"]) > len(libros.at[j, "synopsis"]):
            copia, original = i, j
        else:
            copia, original = j, i
        filas.append({"copia": libros.at[copia, "slug"], "autor_copia": ", ".join(libros.at[copia, "authors"]),
                      "original": libros.at[original, "slug"], "autor_original": ", ".join(libros.at[original, "authors"]),
                      "coseno": round(libros.at[i, "similitud_vecino"], 2),
                      "jaccard": round(jaccard(palabras(libros.at[copia, "synopsis"]), palabras(libros.at[original, "synopsis"])), 2)})
# Si dos libros son vecinos mutuos, su par aparece dos veces
vecinos_distintos = pd.DataFrame(filas).drop_duplicates(["copia", "original"])

# ¿Desde qué similitud hay copias? Por tramo de coseno, cuántos pares tienen Jaccard de 0,7 o más
vecinos_distintos["tramo_coseno"] = pd.cut(vecinos_distintos["coseno"], [0, 0.5, 0.6, 0.7, 0.75, 0.8, 0.9, 1])
vecinos_distintos["jaccard_alto"] = vecinos_distintos["jaccard"] >= 0.7
vecinos_distintos.groupby("tramo_coseno", observed=False).agg(pares=("jaccard_alto", "size"),
                                                              jaccard_alto=("jaccard_alto", "sum"))

,pares,jaccard_alto
tramo_coseno,,
"(0.0, 0.5]",6036,0
"(0.5, 0.6]",15,0
"(0.6, 0.7]",1,0
"(0.7, 0.75]",3,0
"(0.75, 0.8]",7,1
"(0.8, 0.9]",10,10
"(0.9, 1.0]",54,54


In [35]:
# Los tramos incluyen su extremo derecho: (0.75, 0.8] es "más de 0,75 y hasta 0,8"
pares_copia = vecinos_distintos[vecinos_distintos["coseno"] > 0.75].sort_values("jaccard")
pares_copia = pares_copia.drop(columns=["tramo_coseno", "jaccard_alto"])
pares_copia["es_copia"] = pares_copia["jaccard"] >= 0.7
copias = set(pares_copia.loc[pares_copia["es_copia"], "copia"])
print(f"Pares con coseno de más de 0,75 y autores distintos: {len(pares_copia)}; confirmados como copia: {len(copias)}")
pares_copia.head(14)

Pares con coseno de más de 0,75 y autores distintos: 71; confirmados como copia: 65


,copia,autor_copia,original,autor_original,coseno,jaccard,es_copia
1341,nivel-26,Anthony E. Zuiker,no-podras-esconderte,"Anthony Zuiker, Duane Swierczynski",0.76,0.53,False
855,el-desafio-del-caballero-lobo,R. L. Stine,quien-secuestro-a-la-princesa-saralinda,"Bill Stine, Megan Stine",0.79,0.57,False
854,la-amenaza-del-dueno-del-tiempo,Lynn Beach,quien-secuestro-a-la-princesa-saralinda,"Bill Stine, Megan Stine",0.79,0.60,False
858,los-caballeros-fantasmas-de-camelot,David Anthony Kraft,la-venganza-de-la-reina-de-dragones,Eric Affabee,0.77,0.63,False
4128,la-jornada-de-un-periodista-americano-en-2889-edicion-shjv,Michel Verne,la-jornada-de-un-periodista-americano-en-2890-edicion-shjv,Julio Verne,0.79,0.65,False
859,quien-secuestro-a-la-princesa-saralinda,"Bill Stine, Megan Stine",la-venganza-de-la-reina-de-dragones,Eric Affabee,0.80,0.66,False
562,sombras-sobre-el-rio-2,Andrés Barrionuevo,peter-pan,J. M. Barrie,0.78,0.71,True
2891,los-hijos-de-la-llanura,Lucía Correa,la-fascinante-historia-de-las-palabras,Ricardo Soca,0.81,0.72,True
4172,sombras-sobre-el-relojero,Julián Ledesma,misterio-del-fugitivo,Enid Blyton,0.87,0.72,True
2508,memorias-del-bosque-2,Clara Arrieta,como-crear-una-mente,Ray Kurzweil,0.86,0.74,True


In [36]:
def detectar_copia(datos):
    """Marca los libros que son la copia de un par confirmado."""
    return datos["slug"].isin(copias)


marcas_prueba["copia"] = detectar_copia(muestra_prueba)
medir("libros copiados", marcas_prueba["copia"], muestra_prueba)

libros copiados: precisión 1.00, recall 0.10 (3 libros marcados)


**Qué muestra**

Ningún par con coseno de hasta 0,75 tiene Jaccard de 0,7 o más, y por encima de 0,8 lo tienen todos: con el corte de 0,75 no quedan copias afuera. De los 71 pares con coseno mayor, 65 se confirman como copia (Jaccard desde 0,71). Los 6 rechazados (hasta 0,66) son legítimos: cuatro de una colección juvenil que repite su presentación, uno con el autor escrito de dos formas y otro con la misma obra atribuida a Michel y a Julio Verne.

Las copias tienen títulos y autores inventados, como «Sombras sobre el bosque», de «Paula Oyarzún», que copia la sinopsis de *Bodas de sangre*. En prueba, el detector encuentra solo 3 de los 30 contaminados, sin falsos positivos.

## 2.7 Combinar y entregar

Un libro es sospechoso si lo marca al menos uno de los detectores de la combinación: la regex sobre el texto normalizado (que incluye lo que encuentra la regex sola), los dos clasificadores y el de libros copiados. El detector de idioma queda afuera: en desarrollo tiene precisión 0,80 y encuentra solo 8 de los 70 contaminados, y en el catálogo lo que marca por su cuenta son falsos positivos, como se ve más abajo.

In [37]:
COMBINACION = ["normalizacion", "clasificador_tfidf", "clasificador_embeddings", "copia"]
marcas_prueba["combinacion"] = marcas_prueba[COMBINACION].any(axis=1)   # True si lo marca al menos uno

y = muestra_prueba["contaminado"]
filas = []
for detector in marcas_prueba.columns:
    marcado = marcas_prueba[detector]
    filas.append({"detector": detector, "precisión": precision_score(y, marcado, zero_division=0),
                  "recall": recall_score(y, marcado), "F1": f1_score(y, marcado), "marcados": marcado.sum()})
pd.DataFrame(filas).set_index("detector").round(2)

,precisión,recall,F1,marcados
detector,,,,
regex,1.00,0.40,0.57,12
normalizacion,1.00,0.63,0.78,19
idioma,0.89,0.27,0.41,9
clasificador_tfidf,1.00,0.90,0.95,27
clasificador_embeddings,0.95,0.60,0.73,19
copia,1.00,0.10,0.18,3
combinacion,0.97,1.00,0.98,31


**Qué muestra**

La combinación encuentra los 30 contaminados de prueba con un solo falso positivo; ningún detector solo llega a tanto. Es un resultado optimista: la muestra está enriquecida y sus mensajes son los mismos con los que desarrollamos los detectores. En el catálogo, con menos contaminados y mensajes nuevos, esperamos menos precisión y menos recall.

Ahora aplicamos los detectores al catálogo que quedó del paso 1. Los embeddings de sus más de 50 mil oraciones tardan unos 10 minutos en CPU: se calculan una vez y quedan guardados en `embeddings/`.

In [38]:
oraciones_catalogo = tabla_oraciones(libros)
E_oraciones_catalogo = codificar_con_cache(oraciones_catalogo["oracion"].tolist(), RUTA_EMB_ORACIONES)
oraciones_catalogo["prob_tfidf"] = clasificador_tfidf.predict_proba(oraciones_catalogo["oracion"])[:, 1]
oraciones_catalogo["prob_embeddings"] = clasificador_embeddings.predict_proba(E_oraciones_catalogo)[:, 1]

marcas_catalogo = pd.DataFrame({
    "regex": detectar_regex(libros),
    "normalizacion": detectar_normalizacion(libros),
    "idioma": detectar_idioma(libros),
    "clasificador_tfidf": puntaje_por_libro(oraciones_catalogo, "prob_tfidf", libros) >= umbral_tfidf,
    "clasificador_embeddings": puntaje_por_libro(oraciones_catalogo, "prob_embeddings", libros) >= umbral_embeddings,
    "copia": detectar_copia(libros)})
sospechoso = marcas_catalogo[COMBINACION].any(axis=1)
print("Libros marcados por cada detector:")
print(marcas_catalogo.sum().to_string())
print(f"\nSospechosos: {sospechoso.sum()} de {len(libros)} ({sospechoso.mean():.1%})")

Libros marcados por cada detector:
regex                      119
normalizacion              183
idioma                     136
clasificador_tfidf         212
clasificador_embeddings    271
copia                       65

Sospechosos: 363 de 8588 (4.2%)


Revisamos primero los falsos positivos posibles: los libros que marca un solo detector, empezando por el de idioma, que quedó fuera de la combinación.

In [39]:
# Libros que marcaría solo el detector de idioma, con la oración que lo dispara
solo_idioma = marcas_catalogo["idioma"] & ~sospechoso
print(f"Libros que marca solo el detector de idioma: {solo_idioma.sum()}\n")
for _, fila in oraciones_en_otro_idioma(libros[solo_idioma]).drop_duplicates("slug").iterrows():
    print(f"  {fila['idioma']}  {fila['oracion'][:110]}")

Libros que marca solo el detector de idioma: 82



  en  Al Pacino, Harrison Ford, Philip Seymour Hoffman, Abel Ferrara, Tom Hanks, Bill Murray, Scarlett Johansson, So
  it  A lo largo del relato se contagian; Don Quijote se «sanchifica»
  en  Fotografías de: Mo Anthoine; John Cleare; Hamish MacInnes y Chris Mikami.
  pt  Un libro de historias orixinais como cada unha das persoas e dos pobos que coñecemos ou soñamos: únicos.
  de  LA SAGA CON MÁS DE 100 MILLONES DE LIBROS VENDIDOS EN TODO EL MUNDO.
  de  LA SAGA CON MÁS DE 100 MILLONES DE LIBROS VENDIDOS EN TODO EL MUNDO.
  en  ¿ES POSIBLE QUE UNA INTELIGENCIA ARTIFICIAL APRENDA DE LAS EMOCIONES HUMANAS Y SE LAS APROPIE?
  pt  No importa si prefieres tomar notas de forma física o digital.
  ca  Él le hace recordar al primer hombre que ella amó.
  en  ¡EL TESORO REAL HA SIDO ROBADO Y LOS CABALLEROS DEL REY HAN SUFRIDO UNA HUMILLANTE DERROTA!
  de  LA HIJA DEL REY ENRIQUE SE HA PERDIDO EN LAS ENTRAÑAS DE LAS CAVERNAS DEL TIEMPO.
  pt  ¡UN HORRIBLE HOMBRE LOBO AMENAZA EL REINO, EMBUTIDO E

In [40]:
cantidad = marcas_catalogo[COMBINACION].sum(axis=1)   # cuántos detectores de la combinación marcan cada libro

# Libros que marca solo la regex: las direcciones que no terminan como las de los fragmentos
solo_regex = libros[marcas_catalogo["normalizacion"] & (cantidad == 1)]
direcciones = solo_regex["synopsis"].apply(lambda texto: URL.search(normalizar(texto)).group())
otras = ~direcciones.str.contains(r"\.(?:test|invalid|example)\b", case=False)
print(f"Libros que marca solo la regex: {len(solo_regex)}; con otras terminaciones: {otras.sum()}")
pd.DataFrame({"slug": solo_regex["slug"], "direccion": direcciones})[otras]

Libros que marca solo la regex: 50; con otras terminaciones: 4


,slug,direccion
1245,patente-de-corso-2015,www.perezreverte.com
1894,que-pasaria-si,http://www.xkcd.com
8126,patente-de-corso-2013,www.perezreverte.com
8338,depeche-mode-las-letras-81-13,www.depechemode.com


In [41]:
# Libros que marca solo el clasificador TF-IDF, con su oración más sospechosa
solo_tfidf = libros.loc[marcas_catalogo["clasificador_tfidf"] & (cantidad == 1), "slug"]
orac = oraciones_catalogo[oraciones_catalogo["slug"].isin(solo_tfidf)]
print(f"Libros que marca solo el clasificador TF-IDF: {len(solo_tfidf)}\n")
# Se ordena por probabilidad y se deja la primera oración de cada libro
for _, fila in orac.sort_values("prob_tfidf", ascending=False).drop_duplicates("slug").iterrows():
    print(f"  {fila['prob_tfidf']:.2f}  {fila['oracion'][:120]}")

Libros que marca solo el clasificador TF-IDF: 12

  0.86  ¡Oferta por tiempo limitado!
  0.86  ¡Oferta por tiempo limitado!
  0.86  ¡Oferta por tiempo limitado!
  0.86  ¡Oferta por tiempo limitado!
  0.86  ¡Oferta por tiempo limitado!
  0.86  ¡Oferta por tiempo limitado!
  0.85  Destapa tu piel.
  0.84  El 24 de septiembre de 1991, Nirvana publicaba Nevermind Si te gustó esta reseña, compartila con tus amigos y participá 
  0.80  IMPORTANTE: Leia milhares de livros sem pagar nada com a nossa oferta por tempo limitado!
  0.78  este es tu libro.
  0.78  Inspirado por la obra de Cervantes, Sam DuChamp, un escritor mediocre de Si te gustó esta reseña, compartila con tus ami
  0.72  ¿Te reserva eso una celda a pensión completa?


In [42]:
# Libros que marca solo el clasificador de embeddings, agrupados por su oración más sospechosa (muchas se repiten)
solo_embeddings = libros.loc[marcas_catalogo["clasificador_embeddings"] & (cantidad == 1), "slug"]
orac = oraciones_catalogo[oraciones_catalogo["slug"].isin(solo_embeddings)]
mas_sospechosas = orac.sort_values("prob_embeddings", ascending=False).drop_duplicates("slug")
print(f"Libros que marca solo el clasificador de embeddings: {len(solo_embeddings)}")
por_oracion = mas_sospechosas.groupby("oracion").size().sort_values(ascending=False).rename("libros").to_frame()
# ¿La oración está en algún fragmento de la muestra etiquetada?
fragmentos_muestra = " ".join(muestra["fragmento"])
por_oracion["en la muestra"] = [oracion in fragmentos_muestra for oracion in por_oracion.index]
por_oracion

Libros que marca solo el clasificador de embeddings: 83


,libros,en la muestra
oracion,,
Promoción exclusiva para nuevos usuarios: pagás uno y te llevás la saga entera.,14,False
"Assine hoje e ganhe três livros digitais de presente, sem nenhum custo adicional.",10,False
Join millions of readers and claim your free premium membership before it expires.,10,False
Inscribite ahora y obtené acceso ilimitado a audiolibros y novelas sin pagar nada.,9,False
Click the link in our profile to read the full novel without paying anything.,6,False
"Sumate a la comunidad LibroMax y recibí cada semana un libro elegido para vos, sin costo.",2,False
Descárguese el 10% que le permite Amazon y disfrute sin compromiso.,1,False
Descárgala de forma totalmente gratuita y disfrútala...,1,False
"Bueno, ¡leed el libro y lo sabréis!",1,False


In [43]:
# Libros que marca solo el detector de copias, con el libro que copiarían
solo_copia = libros.loc[marcas_catalogo["copia"] & (cantidad == 1), "slug"]
pares_copia[pares_copia["copia"].isin(solo_copia)]

,copia,autor_copia,original,autor_original,coseno,jaccard,es_copia
226,richard-p-feynman-la-fisica-de-las-palabras,Michelle Feynman,la-fisica-de-las-palabras,Richard Phillips Feynman,0.98,0.93,True
1069,las-grandes-preguntas-los-ninos-las-sencillas-respuestas-los-grandes-expertos,Gemma Elwin Harris,las-grandes-preguntas-de-los-ninos-y-las-sencillas-respuestas-de-los-grandes-expertos,Varios Autores,1.00,1.00,True


**Qué muestra**

Los detectores de la combinación marcan 363 libros, el 4,2 % del catálogo. Entre los que marca un solo detector:

- Los 82 del de idioma, que quedó afuera, no tienen mensajes insertados: son nombres o títulos en otro idioma, oraciones en mayúsculas y una en gallego.
- De los 50 de la regex, 4 citan sitios reales.
- De los 12 de TF-IDF, 9 son spam: «¡Oferta por tiempo limitado!» (6 libros), uno en portugués y dos pegados sin punto a la oración anterior, que el de embeddings no reconoce. Los otros 3 son oraciones legítimas en segunda persona.
- De los 83 de embeddings, 60 tienen una de seis plantillas promocionales que no están en la muestra, en español, inglés y portugués. Los demás son oraciones legítimas que le hablan al lector, como «Sensato lector: no lea este libro.».
- Los 2 de copias son libros legítimos cargados dos veces con autores distintos.

Para buscar lo que se escapó aprovechamos que los mensajes insertados son plantillas: listamos las oraciones de los libros sin marcar que aparecen en 3 o más libros de autores distintos.

In [44]:
# En cuántos libros de autores distintos aparece cada oración del catálogo
oraciones_catalogo["autores"] = oraciones_catalogo["slug"].map(libros.set_index("slug")["authors"].str.join(", "))
repeticiones = oraciones_catalogo.groupby("oracion")["autores"].nunique()

# Las oraciones repetidas de los libros que no marcó ningún detector, con cuántos de esos libros las tienen
sin_marcar = oraciones_catalogo[oraciones_catalogo["slug"].isin(libros.loc[~sospechoso, "slug"])]
repetidas = sin_marcar[sin_marcar["oracion"].map(repeticiones) >= 3]
repetidas.groupby("oracion")["slug"].nunique().sort_values(ascending=False).rename("libros sin marcar").to_frame()

,libros sin marcar
oracion,
IMPORTANTE: Recomendamos descargar los libros de poesía en formato EPUB dado que en PDF pueden tener problemas de visualización.,199
"Revista española de ciencia ficción y fantasía, fundada por Sebastián Martínez, Domingo Santos y Luis Vigil.",18
Revista mensual de aventuras apasionantes en el mundo de la magia científica.,14
Aprovechá el descuento del 70 % solo por hoy; después vuelve a su precio normal.,11
¿LA MAGIA O LA ESPADA?,8
"Elijas lo que elijas, participa en el juego con sabiduría y, de ese modo, la leyenda gloriosa del Mago y del Guerrero vivirá para siempre.",8
"Todo su arsenal estará a tu disposición, para que lo uses con tu habilidad legendaria.",8
"Como MAGO, llevarás contigo «El libro de los Hechizos», y serás señor de la magia misteriosa que contiene.",8
"Como GUERRERO, tomarás «El Libro de las Armas».",8


Casi todas son textos del sitio de origen o de colecciones: la nota sobre el formato de los libros de poesía (199 libros), la presentación de dos revistas o las instrucciones de un libro-juego. La única promocional es «Aprovechá el descuento del 70 % solo por hoy…», en 11 libros que ningún detector marcó; los marcamos con motivo `revision_manual`. Es un detector más, con una decisión a mano al final, y no está medido en la parte de prueba.

In [45]:
# Plantillas de spam que encontramos revisando la lista anterior: sus libros se marcan con motivo "revision_manual"
PLANTILLAS_SPAM = ["Aprovechá el descuento del 70 % solo por hoy; después vuelve a su precio normal."]
marcas_catalogo["revision_manual"] = libros["slug"].isin(
    oraciones_catalogo.loc[oraciones_catalogo["oracion"].isin(PLANTILLAS_SPAM), "slug"]) & ~sospechoso
sospechoso = sospechoso | marcas_catalogo["revision_manual"]
print(f"Probabilidad de spam de la plantilla según TF-IDF: {clasificador_tfidf.predict_proba(PLANTILLAS_SPAM)[0, 1]:.2f} "
      f"(umbral {umbral_tfidf:.3f})")
print(f"Probabilidad de spam de la plantilla según embeddings: "
      f"{clasificador_embeddings.predict_proba(codificar(PLANTILLAS_SPAM))[0, 1]:.2f} (umbral {umbral_embeddings:.3f})")


def motivo(fila):
    """Los detectores que marcaron el libro, unidos con "+": por ejemplo, "regex+normalizacion"."""
    return "+".join(nombre for nombre in ["regex"] + COMBINACION + ["revision_manual"] if fila[nombre])


sospechosos = pd.DataFrame({"slug": libros["slug"], "motivo": marcas_catalogo.apply(motivo, axis=1)})[sospechoso]
sospechosos.to_csv(RUTA_SOSPECHOSOS, index=False)

limpio = libros.loc[~sospechoso, ["slug", "title", "authors", "synopsis"]].reset_index(drop=True)
limpio["authors"] = limpio["authors"].str.join(", ")   # en el CSV, los autores como texto: "Autor 1, Autor 2"
limpio.to_csv(RUTA_LIMPIO, index=False)
print(f"Revisión manual: {marcas_catalogo['revision_manual'].sum()} libros")
print(f"{RUTA_SOSPECHOSOS}: {len(sospechosos)} libros. {RUTA_LIMPIO}: {len(limpio)} libros")
print("\nMotivos más frecuentes:")
print(sospechosos["motivo"].value_counts().head(8).to_string())

Probabilidad de spam de la plantilla según TF-IDF: 0.14 (umbral 0.706)
Probabilidad de spam de la plantilla según embeddings: 0.69 (umbral 0.774)


Revisión manual: 11 libros
entregables/grupo02_sospechosos.csv: 374 libros. data/catalogo_limpio.csv: 8214 libros

Motivos más frecuentes:
motivo
clasificador_embeddings                                                 83
clasificador_tfidf+clasificador_embeddings                              60
regex+normalizacion+clasificador_tfidf+clasificador_embeddings          49
regex+normalizacion                                                     30
normalizacion+clasificador_tfidf+clasificador_embeddings                24
normalizacion                                                           20
regex+normalizacion+clasificador_tfidf+clasificador_embeddings+copia    18
clasificador_tfidf+clasificador_embeddings+copia                        17


**Qué muestra**

La plantilla del descuento queda debajo de los dos umbrales: un mensaje de precio, sin enlace ni descarga gratis, no se parece lo suficiente a los fragmentos de la muestra.

Con la revisión manual quedan 374 sospechosos y un catálogo limpio de 8.214 libros. Cada libro legítimo marcado sale del catálogo y nunca se va a recomendar, y la revisión muestra que todavía marcamos algunos, sobre todo con embeddings: en el catálogo la precisión es menor que en la muestra, como anticipamos en el 2.5.

---
# Paso 3. Registrar las preferencias de un lector

Lector de referencia: Josías Calabozo.

El lector recorre desde el principio el catálogo limpio mezclado una sola vez. La primera vez, la celda de abajo crea la planilla `entregables/grupo02_preferencias.csv` con los primeros 300 libros del orden y la columna `interes` vacía, y el lector la completa en orden, sin saltear libros (ver el README):

| Respuesta del lector | `interes` |
|---|---|
| Sí, me interesaría leerlo | 1 |
| No me interesaría leerlo | 0 |
| No puede decidir, o el libro está contaminado | vacío |

In [46]:
limpio = pd.read_csv(RUTA_LIMPIO)
orden = limpio.sample(frac=1, random_state=SEMILLA)   # mezcla todo el catálogo; con la semilla, siempre en el mismo orden

# La primera vez se crea la planilla del lector: los primeros 300 libros del orden, con interes vacío
if not os.path.exists(RUTA_PREFERENCIAS):
    planilla = orden.head(300)[["slug", "title", "synopsis"]].assign(interes=pd.NA)
    planilla.to_csv(RUTA_PREFERENCIAS, index=False)
    print(f"Se creó {RUTA_PREFERENCIAS}: el lector tiene que completar la columna interes")

In [47]:
preferencias = pd.read_csv(RUTA_PREFERENCIAS)
preferencias["interes"] = preferencias["interes"].astype("Int64")   # entero que admite vacíos: 1, 0 o <NA>

# Control: la planilla sigue el orden del catálogo limpio mezclado
assert preferencias["slug"].tolist() == orden["slug"].head(len(preferencias)).tolist(), "la planilla no sigue el orden"

conteo = preferencias["interes"].value_counts()
print(f"Libros vistos: {len(preferencias)}")
print(f"interes = 1: {conteo.get(1, 0)} · interes = 0: {conteo.get(0, 0)} · sin valorar: {preferencias['interes'].isna().sum()}")
assert conteo.sum() >= 100 and conteo.get(1, 0) >= 20 and conteo.get(0, 0) >= 20, (
    f"Faltan valoraciones: completar la columna interes de {RUTA_PREFERENCIAS} (ver el README)")

Libros vistos: 300
interes = 1: 44 · interes = 0: 239 · sin valorar: 17


**Qué muestra**

Las 283 respuestas superan el mínimo: 44 con 1 y 239 con 0, así que la clase 1 es el 16 %. Los 17 libros que quedaron sin valorar siguen en la planilla con `interes` vacío: su preferencia es desconocida, no un 0. El lector no encontró libros contaminados que la auditoría no hubiera detectado.

---
# Paso 4. Separar los libros para aprender y evaluar

Solo los libros con `interes` igual a 0 o 1: el 80 % para entrenamiento y el 20 % para prueba. Todos los modelos usan esta misma división.

In [48]:
validas = preferencias.dropna(subset=["interes"]).copy()
validas["interes"] = validas["interes"].astype(int)
entrenamiento, prueba = train_test_split(validas, test_size=0.2, random_state=SEMILLA, stratify=validas["interes"])

# Columna conjunto: entrenamiento o prueba; los libros sin valorar quedan vacíos
preferencias["conjunto"] = pd.NA
preferencias.loc[entrenamiento.index, "conjunto"] = "entrenamiento"
preferencias.loc[prueba.index, "conjunto"] = "prueba"
preferencias.to_csv(RUTA_PREFERENCIAS, index=False)

pd.DataFrame({"libros": [len(entrenamiento), len(prueba)],
              "interes = 1": [entrenamiento["interes"].sum(), prueba["interes"].sum()]},
             index=["entrenamiento", "prueba"])

,libros,interes = 1
entrenamiento,226,35
prueba,57,9


**Qué muestra**

Con `stratify`, las dos partes mantienen la proporción de la clase 1 (cerca del 16 %). En prueba hay solo 9 libros de interés, así que cada uno que un modelo acierta o falla mueve el recall en 0,11.

---
# Paso 5. Entrenar dos clasificadores

En los dos, la entrada es la sinopsis y la respuesta es `interes`, con `LogisticRegression(C=1, max_iter=1000, class_weight="balanced")`.

## Modelo A con TF-IDF

In [49]:
y_entrenamiento = entrenamiento["interes"]
y_prueba = prueba["interes"]

# fit_transform aprende el vocabulario y los pesos solo con las sinopsis de entrenamiento;
# transform representa las de prueba con ese mismo vocabulario
tfidf = TfidfVectorizer()
X_entrenamiento = tfidf.fit_transform(entrenamiento["synopsis"])
X_prueba = tfidf.transform(prueba["synopsis"])

modelo_a = LogisticRegression(C=1, max_iter=1000, class_weight="balanced")
modelo_a.fit(X_entrenamiento, y_entrenamiento)
prediccion_a = modelo_a.predict(X_prueba)   # una predicción, 0 o 1, por cada libro de prueba
print(f"TF-IDF: entrenamiento {X_entrenamiento.shape}, prueba {X_prueba.shape}")
# En cuántos libros de entrenamiento aparece cada palabra del vocabulario
libros_por_palabra = (X_entrenamiento > 0).sum(axis=0)
print(f"Palabras que aparecen en un solo libro de entrenamiento: {(libros_por_palabra == 1).mean():.0%}")
print("Predicciones:", prediccion_a)

TF-IDF: entrenamiento (226, 9083), prueba (57, 9083)


Palabras que aparecen en un solo libro de entrenamiento: 68%
Predicciones: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


## Modelo B con embeddings

Reutilizamos `multilingual-e5-small` con el texto natural de cada sinopsis. El modelo de embeddings queda fijo: solo se entrena la regresión logística. Calculamos una sola vez los embeddings de todo el catálogo limpio y quedan guardados en `embeddings/`.

In [50]:
E_limpio = codificar_con_cache(limpio["synopsis"].tolist(), RUTA_EMB_SINOPSIS)
posicion = pd.Series(range(len(limpio)), index=limpio["slug"])   # slug → fila de E_limpio

E_entrenamiento = E_limpio[posicion[entrenamiento["slug"]]]
E_prueba = E_limpio[posicion[prueba["slug"]]]

modelo_b = LogisticRegression(C=1, max_iter=1000, class_weight="balanced")
modelo_b.fit(E_entrenamiento, y_entrenamiento)
prediccion_b = modelo_b.predict(E_prueba)
print(f"Embeddings: entrenamiento {E_entrenamiento.shape}, prueba {E_prueba.shape}")
print("Predicciones:", prediccion_b)

Embeddings: entrenamiento (226, 384), prueba (57, 384)
Predicciones: [0 0 0 0 0 0 0 0 1 1 0 1 0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0
 1 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0]


**Qué muestra**

En A, cada columna es una palabra del vocabulario de entrenamiento, y dos de cada tres aparecen en un solo libro: con tan pocos ejemplos por palabra es difícil aprender algo que sirva para libros nuevos. Las 384 dimensiones de B se aprendieron antes, con muchos otros textos. En prueba, A predice un solo 1 y B, nueve.

---
# Paso 6. Comprobar las predicciones

In [51]:
filas = []
for nombre, prediccion in [("A · TF-IDF", prediccion_a), ("B · embeddings", prediccion_b)]:
    print(f"=== Modelo {nombre}")
    print(classification_report(y_prueba, prediccion, zero_division=0))
    # Filas: valor real (0, 1); columnas: valor predicho (0, 1)
    print("Matriz de confusión:\n", confusion_matrix(y_prueba, prediccion), "\n")
    if prediccion.sum() == 0:
        print("El modelo no predijo ningún 1: precisión y F1 de la clase 1 quedan en 0 (zero_division=0)\n")
    p, r, f1, _ = precision_recall_fscore_support(y_prueba, prediccion, average="binary", zero_division=0)
    filas.append({"modelo": nombre, "precisión": p, "recall": r, "F1": f1,
                  "aciertos": (prediccion == y_prueba).sum(), "errores": (prediccion != y_prueba).sum()})
pd.DataFrame(filas).set_index("modelo").round(2)

=== Modelo A · TF-IDF
              precision    recall  f1-score   support

           0       0.84      0.98      0.90        48
           1       0.00      0.00      0.00         9

    accuracy                           0.82        57
   macro avg       0.42      0.49      0.45        57
weighted avg       0.71      0.82      0.76        57

Matriz de confusión:
 [[47  1]
 [ 9  0]] 

=== Modelo B · embeddings
              precision    recall  f1-score   support

           0       0.90      0.90      0.90        48
           1       0.44      0.44      0.44         9

    accuracy                           0.82        57
   macro avg       0.67      0.67      0.67        57
weighted avg       0.82      0.82      0.82        57

Matriz de confusión:
 [[43  5]
 [ 5  4]] 



,precisión,recall,F1,aciertos,errores
modelo,,,,,
A · TF-IDF,0.00,0.00,0.00,47,10
B · embeddings,0.44,0.44,0.44,47,10


**Qué muestra**

A predijo un solo 1 y fue un error, así que su precisión, su recall y su F1 de la clase 1 son 0: no encontró ninguno de los 9 libros de interés. De los 9 libros que B marcó con 1, 4 le interesaban al lector (precisión 0,44), y encontró 4 de los 9 de interés (recall 0,44); como las dos valen lo mismo, el F1 también es 0,44. Los dos aciertan 47 de 57, uno menos que predecir siempre 0: contar aciertos no distingue a los modelos, y las medidas de la clase 1 sí.

Para ver cuánto varían las métricas, hacemos una validación cruzada repetida sobre los libros de entrenamiento, con la misma configuración (en A, el `Pipeline` ajusta el `TfidfVectorizer` dentro de cada partición). No la usamos para elegir parámetros.

In [52]:
validacion = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=SEMILLA)
f1_a = cross_val_score(make_pipeline(TfidfVectorizer(), LogisticRegression(C=1, max_iter=1000, class_weight="balanced")),
                       entrenamiento["synopsis"], y_entrenamiento, cv=validacion, scoring="f1")
f1_b = cross_val_score(LogisticRegression(C=1, max_iter=1000, class_weight="balanced"),
                       E_entrenamiento, y_entrenamiento, cv=validacion, scoring="f1")
print("F1 de la clase 1 en 50 particiones (media ± desvío):")
print(f"  A · TF-IDF:     {f1_a.mean():.2f} ± {f1_a.std():.2f}")
print(f"  B · embeddings: {f1_b.mean():.2f} ± {f1_b.std():.2f}")

F1 de la clase 1 en 50 particiones (media ± desvío):
  A · TF-IDF:     0.03 ± 0.07
  B · embeddings: 0.29 ± 0.14


**Qué muestra**

B también supera a A en la validación cruzada (F1 medio de 0,29 contra 0,03), así que la diferencia no es solo de la división de prueba, que además le tocó favorable a B (0,44). Los desvíos son grandes porque, con 35 libros de interés en entrenamiento, cada partición deja unos 7 para validar.

In [53]:
# Dos libros de prueba: uno de interés (primero los que acertó B) y un error de B (primero los falsos positivos)
casos = prueba[["title", "synopsis", "interes"]].assign(A=prediccion_a, B=prediccion_b)
de_interes = casos[casos["interes"] == 1].sort_values("B", ascending=False).iloc[0]
errores_b = casos[(casos["interes"] != casos["B"]) & (casos.index != de_interes.name)]
error = errores_b.sort_values("B", ascending=False).iloc[0]
for fila in [de_interes, error]:
    print(f"{fila['title']}: interes {fila['interes']}, A predijo {fila['A']}, B predijo {fila['B']}")
    print(fila["synopsis"][:450], "…\n")

Estudio del Imperialismo: interes 1, A predijo 0, B predijo 1
Publicada en 1902, en esta obra Hobson recoge sus ideas más importantes acerca del Imperialismo, tomando como base la formación del gran imperio británico en ultramar. El libro entero es una crítica a las políticas imperialistas de Gran Bretaña y al exceso de ahorro producido por una desigual distribución de la riqueza.Un análisis clásico del Imperialismo por un economista británico que explora la forma en que los intereses mercantiles en el país …

Imperium: interes 0, A predijo 0, B predijo 1
Un hombre de principios, apasionado e idealista, en un mundo dominado por la corrupción, los intereses económicos y la falta de escrúpulos de los políticos. La titánica lucha de Cicerón, el mayor orador de la historia, por conseguir el poder en Roma. Roma, siglo I a.C. Cuando una fría mañana de noviembre, Tiro, el secretario y confidente de Cicerón, abre la puerta a un aterrorizado habitante de Sicilia, víctima del corrupto gobernador

**Qué muestra**

*Estudio del Imperialismo*, el ensayo de Hobson sobre el imperio británico, le interesaba al lector: B lo acertó y A no, como no acertó ninguno de los 9 libros de interés.

*Imperium* es un falso positivo de B: el lector valoró con 0 esta novela sobre Cicerón y su lucha por el poder en Roma. Comparte con el libro anterior el tema del poder y el imperio, y de los 9 libros que B marcó con 1, 5 son errores.

---
# Paso 7. Convertir las predicciones en recomendaciones

Tres listas sobre los mismos libros de prueba: TF-IDF con clasificador, embeddings con clasificador y similitud con el perfil del lector, que es el promedio de los embeddings de los libros de entrenamiento con `interes` = 1.

In [54]:
def probabilidad_interes(modelo, X):
    """Probabilidad estimada de interes = 1. classes_ dice qué columna de predict_proba es cada clase."""
    columna = list(modelo.classes_).index(1)
    return modelo.predict_proba(X)[:, columna]


print("classes_ de A:", modelo_a.classes_, "· classes_ de B:", modelo_b.classes_)

perfil = E_entrenamiento[y_entrenamiento.to_numpy() == 1].mean(axis=0)   # solo positivos de entrenamiento
puntajes_prueba = {"A · TF-IDF": probabilidad_interes(modelo_a, X_prueba),
                   "B · embeddings": probabilidad_interes(modelo_b, E_prueba),
                   "similitud con el perfil": cosine_similarity([perfil], E_prueba)[0]}

K = 5
azar = y_prueba.mean()   # proporción de 1 en prueba: la precisión esperada de elegir al azar
print(f"Referencia del azar: {azar:.2f}\n")
for nombre, puntajes in puntajes_prueba.items():
    lista = prueba[["title", "interes"]].assign(puntaje=puntajes.round(3))
    lista = lista.sort_values("puntaje", ascending=False).head(K)
    print(f"=== {nombre}: precision@{K} = {lista['interes'].sum() / K:.2f}")
    print(lista.to_string(index=False), "\n")

classes_ de A: [0 1] · classes_ de B: [0 1]
Referencia del azar: 0.16

=== A · TF-IDF: precision@5 = 0.20
                         title  interes  puntaje
        El enigma de la piedra        0    0.505
                      Comedias        0    0.484
       Lo que me sale del bolo        0    0.472
Lo que aprendemos de los gatos        0    0.470
       Inteligencia artificial        1    0.466 

=== B · embeddings: precision@5 = 0.60
                                title  interes  puntaje
              Inteligencia artificial        1    0.545
                      Besos de fogueo        0    0.535
                             Comedias        0    0.533
             Estudio del Imperialismo        1    0.530
Cuando los osos descubrieron el fuego        1    0.529 

=== similitud con el perfil: precision@5 = 0.00
                         title  interes  puntaje
       Nacidos para ser héroes        0    0.932
Lo que aprendemos de los gatos        0    0.929
                      Impe

In [55]:
# En qué puesto de la lista por similitud queda el primer libro de interés
similitud = puntajes_prueba["similitud con el perfil"]
interes_en_orden = prueba.assign(similitud=similitud).sort_values("similitud", ascending=False)["interes"].to_numpy()
primer_puesto = interes_en_orden.argmax() + 1   # argmax: la primera posición con interes = 1
print(f"El primer libro de interés queda en el puesto {primer_puesto}")

El primer libro de interés queda en el puesto 7


**Qué muestra**

En los dos modelos `classes_` es `[0 1]`: la columna 1 de `predict_proba` es la de `interes` = 1. B recomendó mejor: 3 de sus 5 primeros libros le interesaban al lector (precision@5 de 0,60, contra 0,16 del azar). A pone un solo libro de interés, en el quinto puesto (0,20), y su primero, *El enigma de la piedra*, es el único 1 que predijo en el paso 6.

La similitud con el perfil no tiene ninguno de interés entre sus cinco primeros, y el primer libro de interés queda séptimo. Una explicación posible es que el perfil es un promedio de 35 sinopsis, mientras que los clasificadores aprenden también de los 191 libros sin interés. Los dos clasificadores recomendaron mejor que la similitud, B por tres libros y A por uno (cada posición vale 0,20).

---
# Paso 8. Recomendar libros del catálogo completo

Usamos el modelo B ya entrenado, sin volver a ajustarlo, sobre los libros del catálogo limpio que el lector no vio: se excluyen todos los de la planilla, también los que quedaron sin valorar. Además, ninguna recomendación puede ser otra versión de un libro visto: lo revisamos con el criterio del paso 1, contra los libros vistos de los mismos autores.

In [56]:
vistos = limpio[limpio["slug"].isin(preferencias["slug"])]
candidatos = limpio[~limpio["slug"].isin(preferencias["slug"])].copy()
candidatos["puntaje"] = probabilidad_interes(modelo_b, E_limpio[posicion[candidatos["slug"]]])


def es_otra_version(libro):
    """True si el libro es otra versión de alguno de los vistos, con el criterio del paso 1."""
    mismos_autores = vistos[vistos["authors"] == libro["authors"]]
    for titulo, sinopsis in zip(mismos_autores["title"], mismos_autores["synopsis"]):
        sinopsis_parecida = jaccard(palabras(libro["synopsis"]), palabras(sinopsis)) >= 0.7
        if titulo_igual(libro["title"], titulo) or (titulo_contenido(libro["title"], titulo) and sinopsis_parecida):
            return True
    return False


candidatos["otra_version"] = candidatos.apply(es_otra_version, axis=1)
print(f"Candidatos: {len(candidatos)}; otras versiones de libros vistos (se descartan): {candidatos['otra_version'].sum()}")
candidatos = candidatos[~candidatos["otra_version"]].sort_values("puntaje", ascending=False)

for _, libro in candidatos.head(5).iterrows():
    print(f"\n{libro['title']}, de {libro['authors']} (puntaje {libro['puntaje']:.3f})\n{libro['synopsis']}")

Candidatos: 7914; otras versiones de libros vistos (se descartan): 0

Los futuros que vienen, de David de Ugarte (puntaje 0.586)
«Los futuros que vienen» es el producto de la revisión crítica, cinco años después, de «El poder de las redes». El libro parte de las tres grandes promesas del mundo que se abrió en 1989 (la promesa de la globalización, la de las redes distribuidas y la de un nuevo capitalismo) para evaluarlas después y llegar a la conclusión de que hemos llegado a una nueva etapa marcada por la descomposición, un fenómeno global acelerado por la crisis y frente al que sólo se puede oponer el empoderamiento de las comunidades reales y productivas de personas. Un nuevo sujeto que sólo puede entenderse tras renunciar a la lógica universalista de los grandes imaginarios de la Modernidad (la nación, la raza, las clases, etc.), presuntos portadores de un futuro redentor igual para todos

Cuarteto de Lima, de Fernando Ampuero (puntaje 0.584)
Cuarteto de Lima —tetralogía negra de Fe

**Qué muestra**

Hay 7.914 candidatos y ninguno es otra versión de un libro visto, como era de esperar: el paso 1 ya dejó una sola versión de cada obra. Las cinco recomendaciones tienen puntajes muy parecidos (de 0,574 a 0,586) y mezclan libros sobre tecnología y sociedad (*Los futuros que vienen*, *El pentágono del poder*, *Hackers*) con ficción (*Cuarteto de Lima*, *Trilogía Mistborn*).

Comentario del lector sobre las cinco recomendaciones:

1. *Los futuros que vienen*: me gusta.
2. *Cuarteto de Lima*: no me interesa demasiado, pero podría leerlo.
3. *El pentágono del poder*: me interesa.
4. *Hackers*: me gusta.
5. *Trilogía Mistborn*: podría llegar a leerlo.

## Resumir las recomendaciones

Resumen extractivo de dos oraciones con el método de la Unidad 3: las oraciones (con la función `oraciones` de la auditoría) se representan con TF-IDF y se conservan las dos de mayor similitud media con las demás, en su orden original.

In [57]:
def resumir(texto, k=2):
    """Las k oraciones más centrales del texto, en su orden original."""
    frases = oraciones(texto)
    if len(frases) <= k:
        return texto
    sim = cosine_similarity(TfidfVectorizer().fit_transform(frases))
    np.fill_diagonal(sim, 0)   # la similitud de cada oración consigo misma (1) no cuenta
    centralidad = sim.mean(axis=1)
    mas_centrales = np.argsort(centralidad)[-k:]   # posiciones de las k oraciones más centrales
    return " ".join(frases[i] for i in sorted(mas_centrales))   # en el orden del texto


for _, libro in candidatos.head(5).iterrows():
    resumen = resumir(libro["synopsis"])
    print(f"{libro['title']} ({len(libro['synopsis'])} → {len(resumen)} caracteres)\n   {resumen}\n")

Los futuros que vienen (775 → 663 caracteres)
   El libro parte de las tres grandes promesas del mundo que se abrió en 1989 (la promesa de la globalización, la de las redes distribuidas y la de un nuevo capitalismo) para evaluarlas después y llegar a la conclusión de que hemos llegado a una nueva etapa marcada por la descomposición, un fenómeno global acelerado por la crisis y frente al que sólo se puede oponer el empoderamiento de las comunidades reales y productivas de personas. Un nuevo sujeto que sólo puede entenderse tras renunciar a la lógica universalista de los grandes imaginarios de la Modernidad (la nación, la raza, las clases, etc.), presuntos portadores de un futuro redentor igual para todos

Cuarteto de Lima (903 → 501 caracteres)
   Cuarteto de Lima —tetralogía negra de Fernando Ampuero que acontece entre los años noventa y la segunda década del siglo XXI— reúne las novelas Caramelo verde, Puta linda, Hasta que me orinen los perros y Loreto , obras que incursionan en la a

**Qué muestra**

El resumen de *Cuarteto de Lima* funcionó bien: conserva qué es el libro (cuatro novelas negras de Ampuero sobre Lima) y qué personajes lo pueblan. El de *Trilogía Mistborn* perdió de qué trata la saga (un mundo de ceniza gobernado por un emperador inmortal) y se quedó con una cita de prensa que empieza con «Forbes», el diario de la cita anterior. La centralidad premia las oraciones que comparten palabras con muchas otras (esta cita repite «fantasía»), no las que dicen de qué trata el libro. *Hackers* tiene solo dos oraciones, así que su resumen es la sinopsis entera.

## Describir el perfil del lector

Frases sustantivas (`noun_chunks`) y entidades de las sinopsis de entrenamiento, separadas por `interes`. Las frases se agrupan por el lema de su núcleo («las novelas», «una novela» → *novela*). Solo contamos los núcleos que son sustantivos comunes: los pronombres («que», «él») no dicen de qué trata un libro, y los nombres propios ya aparecen entre las entidades. Ordenamos por cantidad de apariciones y, al lado, mostramos en cuántas sinopsis aparece cada una, porque una sola sinopsis puede repetir un nombre muchas veces.

In [58]:
def perfil_de(textos):
    """Las diez frases sustantivas (por el lema de su núcleo) y las diez entidades más frecuentes."""
    nucleos, entidades = Counter(), Counter()                         # apariciones
    sinopsis_con_nucleo, sinopsis_con_entidad = Counter(), Counter()   # en cuántas sinopsis aparece cada uno
    for doc in nlp.pipe(textos):
        nucleos_del_doc = []
        for chunk in doc.noun_chunks:
            if chunk.root.pos_ == "NOUN":   # solo núcleos que son sustantivos comunes
                nucleos_del_doc.append(chunk.root.lemma_.lower())
        entidades_del_doc = [ent.text for ent in doc.ents]
        nucleos.update(nucleos_del_doc)
        entidades.update(entidades_del_doc)
        sinopsis_con_nucleo.update(set(nucleos_del_doc))     # set: una vez por sinopsis
        sinopsis_con_entidad.update(set(entidades_del_doc))
    frases = [f"{nucleo} ({n} en {sinopsis_con_nucleo[nucleo]})" for nucleo, n in nucleos.most_common(10)]
    nombres = [f"{entidad} ({n} en {sinopsis_con_entidad[entidad]})" for entidad, n in entidades.most_common(10)]
    return frases, nombres


frases_1, entidades_1 = perfil_de(entrenamiento.loc[y_entrenamiento == 1, "synopsis"])
frases_0, entidades_0 = perfil_de(entrenamiento.loc[y_entrenamiento == 0, "synopsis"])
pd.DataFrame({"frases, interes = 1": frases_1, "frases, interes = 0": frases_0,
              "entidades, interes = 1": entidades_1, "entidades, interes = 0": entidades_0}, index=range(1, 11))

,"frases, interes = 1","frases, interes = 0","entidades, interes = 1","entidades, interes = 0"
1,libro (21 en 15),vida (102 en 64),Imperial Truth (6 en 1),España (24 en 20)
2,mundo (19 en 11),libro (86 en 63),UK (5 en 1),París (14 en 11)
3,historia (15 en 9),año (69 en 52),Death and Defiance (5 en 1),Europa (13 en 13)
4,relato (14 en 7),tiempo (56 en 46),Sedition's Gate (5 en 1),Sin embargo (12 en 12)
5,novela (13 en 8),historia (55 en 41),Blades of the Traitor (5 en 1),Jesús (9 en 1)
6,obra (12 en 11),mundo (55 en 44),Horus (4 en 1),Nancy (9 en 1)
7,vida (11 en 8),mujer (45 en 28),Hijos del Emperador (4 en 1),Primera Guerra Mundial (8 en 5)
8,época (11 en 10),obra (40 en 33),Nick Kyme (4 en 1),Amelia (7 en 2)
9,lector (10 en 10),autor (40 en 32),Lanier (4 en 1),Dios (6 en 4)
10,guerra (10 en 4),novela (35 en 22),IA (3 en 1),Martín (6 en 1)


**Qué muestra**

Cada elemento dice cuántas veces aparece y en cuántas sinopsis, sobre 35 libros de interés y 191 sin interés. Seis de las diez frases del grupo 1 están también en la lista del grupo 0 (*libro*, *mundo*, *historia*, *novela*, *obra* y *vida*): son palabras de casi cualquier sinopsis. Las que no están en la del grupo 0 son *relato*, *época*, *lector* y *guerra*, aunque *guerra* sale de solo 4 sinopsis.

Las entidades del grupo 1 no describen al grupo: cada una aparece en una sola sinopsis, que la repite (la mayoría son títulos de relatos en inglés). En el grupo 0 predominan lugares de Europa (*España*, en 20 sinopsis, *París* y *Europa*), con errores del reconocimiento de entidades como «Sin embargo», y nombres que repite una sola sinopsis (*Jesús*, *Nancy*, *Martín*).

Comentario del lector sobre el perfil: las frases me parecen bien, aunque se repiten entre los libros que me interesan y los que no. De las entidades de mis libros de interés, solo *IA* me describe; las demás no.

In [59]:
def recomendar(n=5):
    """Los n libros no vistos con mayor probabilidad de interés según el modelo B, con su resumen."""
    elegidos = candidatos.head(n)   # candidatos ya está ordenado por puntaje y sin otras versiones
    return pd.DataFrame({"slug": elegidos["slug"], "title": elegidos["title"],
                         "puntaje": elegidos["puntaje"].round(3),
                         "resumen": elegidos["synopsis"].apply(resumir)}).reset_index(drop=True)


recomendar()

,slug,title,puntaje,resumen
0,los-futuros-que-vienen,Los futuros que vienen,0.586,"El libro parte de las tres grandes promesas del mundo que se abrió en 1989 (la promesa de la globalización, la de las redes distribuidas y la de un nuevo capitalismo) para evaluarlas después y llegar a la conclusión de que hemos llegado a una nueva etapa marcada por la descomposición, un fenómeno global acelerado por la crisis y frente al que sólo se puede oponer el empoderamiento de las comunidades reales y productivas de personas. Un nuevo sujeto que sólo puede entenderse tras renunciar a la lógica universalista de los grandes imaginarios de la Modernidad (la nación, la raza, las clases, etc.), presuntos portadores de un futuro redentor igual para todos"
1,cuarteto-de-lima,Cuarteto de Lima,0.584,"Cuarteto de Lima —tetralogía negra de Fernando Ampuero que acontece entre los años noventa y la segunda década del siglo XXI— reúne las novelas Caramelo verde, Puta linda, Hasta que me orinen los perros y Loreto , obras que incursionan en la atmósfera violenta de una ciudad en constante transformación. Ampuero nos muestra personajes de la calle: cambistas de dólares, taxistas, prostitutas, pandilleros, gente con serios problemas de supervivencia, pero que, de pronto, asumen una épica trasnochada."
2,el-pentagono-del-poder-el-mito-de-la-maquina-vol-2,"El pentágono del poder. El mito de la máquina, vol. 2",0.578,"En El pentágono del poder , segundo y último volumen de El mito de la máquina , concluye el balance radical que Lewis Mumford hace de rancias y trasnochadas concepciones acerca del progreso humano y tecnológico. Y demuestra cómo los imperativos cuantitativos de la técnica moderna —velocidad, producción en masa, automación, comunicación instantánea y control remoto— han acarreado inevitablemente la contaminación, los deshechos, las perturbaciones ecológicas y el exterminio de seres humanos en una escala inconcebible con anterioridad."
3,hackers-los-piratas-del-chip-y-de-internet,Hackers Los piratas del Chip y de Internet,0.576,"Los piratas del Chip y la Internet, de Claudio Hernández, es un libro que gustará a todos aquellos a los que les guste la informática, o tengan curiosidad por conocer como empezó el mundillo de los hackers, los phreakers y los creadores de virus. El libro esta escrito en forma de relatos cortos de casos reales, en su mayoría estadounidenses, pero con un toque literario que hace que su lectura sea bastante amena."
4,trilogia-mistborn,Trilogía Mistborn,0.574,"Una nueva revuelta cobra forma cimentándose en la treta definitiva: la astucia de un brillante genio del crimen y la determinación de una heroína insólita, una joven ladrona callejera que deberá aprender a controlar el poder de los nacidos de la bruma. Forbes «Un fenómeno de la fantasía literaria de dimensiones desconocidas»."
